Cell 1：Clone 官方 SpatialGlue + 官方复现 notebooks

In [1]:
# ============================================================
# Cell 1
# Clone official SpatialGlue repositories
# ============================================================

from pathlib import Path
import subprocess
import os


WORK_ROOT = Path(
    "/kaggle/working"
)

SPATIALGLUE_ROOT = (
    WORK_ROOT
    / "SpatialGlue"
)

SPATIALGLUE_NOTEBOOK_ROOT = (
    WORK_ROOT
    / "SpatialGlue_notebook"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


# ------------------------------------------------------------
# Official SpatialGlue
# ------------------------------------------------------------

if not SPATIALGLUE_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/JinmiaoChenLab/SpatialGlue.git",
            str(SPATIALGLUE_ROOT),
        ],
        check=True,
    )

else:

    print(
        "SpatialGlue already exists:",
        SPATIALGLUE_ROOT,
    )


# ------------------------------------------------------------
# Official reproduction notebooks
# ------------------------------------------------------------

if not SPATIALGLUE_NOTEBOOK_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/JinmiaoChenLab/SpatialGlue_notebook.git",
            str(SPATIALGLUE_NOTEBOOK_ROOT),
        ],
        check=True,
    )

else:

    print(
        "SpatialGlue_notebook already exists:",
        SPATIALGLUE_NOTEBOOK_ROOT,
    )


assert SPATIALGLUE_ROOT.exists()
assert SPATIALGLUE_NOTEBOOK_ROOT.exists()
assert DATA_ROOT.exists()


print("=" * 100)
print("PATHS")
print("=" * 100)

print(
    "SpatialGlue         =",
    SPATIALGLUE_ROOT,
)

print(
    "Official notebooks =",
    SPATIALGLUE_NOTEBOOK_ROOT,
)

print(
    "Dataset root        =",
    DATA_ROOT,
)

print(
    "\nPASS: repositories cloned."
)

Cloning into '/kaggle/working/SpatialGlue'...
Cloning into '/kaggle/working/SpatialGlue_notebook'...


PATHS
SpatialGlue         = /kaggle/working/SpatialGlue
Official notebooks = /kaggle/working/SpatialGlue_notebook
Dataset root        = /kaggle/input/datasets/wuvdji/smgc-data

PASS: repositories cloned.


Cell 2：检查 Kaggle 当前环境，不修改版本

In [2]:
# ============================================================
# Cell 2
# Environment inspection
#
# IMPORTANT:
# Do NOT downgrade Python / NumPy yet.
# ============================================================

import sys
import importlib.util


print("=" * 100)
print("CURRENT KAGGLE ENVIRONMENT")
print("=" * 100)

print(
    "Python:",
    sys.version,
)


packages = [
    "torch",
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "anndata",
    "scanpy",
    "skmisc",
    "rpy2",
]


for package in packages:

    spec = importlib.util.find_spec(
        package
    )

    if spec is None:

        print(
            f"{package:12s}: NOT INSTALLED"
        )

        continue


    try:

        module = __import__(
            package
        )

        version = getattr(
            module,
            "__version__",
            "unknown",
        )

        print(
            f"{package:12s}: {version}"
        )

    except Exception as e:

        print(
            f"{package:12s}: "
            f"IMPORT ERROR -> {repr(e)}"
        )


try:

    import torch

    print(
        "\nCUDA available:",
        torch.cuda.is_available(),
    )

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0),
        )

except Exception as e:

    print(
        "Torch inspection failed:",
        repr(e),
    )


print(
    "\nPASS: environment inspection complete."
)

CURRENT KAGGLE ENVIRONMENT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch       : 2.10.0+cu128
numpy       : 2.0.2
pandas      : 2.3.3
scipy       : 1.16.3
sklearn     : 1.6.1
anndata     : NOT INSTALLED
scanpy      : NOT INSTALLED
skmisc      : NOT INSTALLED
rpy2        : 3.5.17

CUDA available: True
GPU: Tesla T4

PASS: environment inspection complete.


Cell 3：检查你的 5 个数据集到底有哪些文件

In [3]:
# ============================================================
# Cell 3
# Inspect SpaMGCL dataset files
# ============================================================

from pathlib import Path


DATASET_TERMS = [
    "HLN",
    "A1",
    "D1",
    "E18",
    "E15",
    "S2",
]


print("=" * 110)
print("DATASET FILE TREE")
print("=" * 110)


all_files = sorted(
    [
        p
        for p in DATA_ROOT.rglob("*")
        if p.is_file()
    ]
)


print(
    "Total files:",
    len(all_files),
)


for path in all_files:

    relative = path.relative_to(
        DATA_ROOT
    )

    text = str(
        relative
    ).lower()

    if any(
        term.lower() in text
        for term in DATASET_TERMS
    ):

        size_mb = (
            path.stat().st_size
            / 1024
            / 1024
        )

        print(
            f"{size_mb:9.2f} MB | "
            f"{relative}"
        )


print(
    "\nPASS: dataset inspection complete."
)

DATASET FILE TREE
Total files: 12
   212.46 MB | E18.5_mouse_brain/adata_ATAC.h5ad
   267.63 MB | E18.5_mouse_brain/adata_RNA.h5ad
     1.09 MB | Human_Lymph_Nodes/A1/adata_ADT.h5ad
    53.95 MB | Human_Lymph_Nodes/A1/adata_RNA.h5ad
     1.06 MB | Human_Lymph_Nodes/D1/adata_ADT.h5ad
    30.14 MB | Human_Lymph_Nodes/D1/adata_RNA.h5ad
   101.44 MB | Mouse_Embryos_S2/E15/adata_ATAC.h5ad
    53.79 MB | Mouse_Embryos_S2/E15/adata_RNA.h5ad
   100.64 MB | Mouse_Embryos_S2/E18/adata_ATAC.h5ad
    51.14 MB | Mouse_Embryos_S2/E18/adata_RNA.h5ad

PASS: dataset inspection complete.


Cell 4：检查 SpatialGlue 实际源码 API

In [4]:
# ============================================================
# Cell 4
# Inspect official SpatialGlue source structure
# ============================================================

from pathlib import Path


print("=" * 110)
print("SPATIALGLUE SOURCE TREE")
print("=" * 110)


for path in sorted(
    SPATIALGLUE_ROOT.rglob("*.py")
):

    relative = path.relative_to(
        SPATIALGLUE_ROOT
    )

    print(relative)


print(
    "\n" + "=" * 110
)

print(
    "IMPORTANT API DEFINITIONS"
)

print(
    "=" * 110
)


KEYWORDS = [
    "class Train",
    "class Encoder",
    "def train",
    "def clustering",
    "def construct_neighbor_graph",
    "def adjacent_matrix_preprocessing",
    "def pca",
    "def lsi",
    "def fix_seed",
    "def clr_normalize",
]


for path in sorted(
    SPATIALGLUE_ROOT.rglob("*.py")
):

    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    lines = text.splitlines()

    hits = []


    for lineno, line in enumerate(
        lines,
        start=1,
    ):

        if any(
            keyword.lower()
            in line.lower()
            for keyword in KEYWORDS
        ):

            hits.append(
                (
                    lineno,
                    line,
                )
            )


    if hits:

        print(
            "\n" + "-" * 100
        )

        print(
            path.relative_to(
                SPATIALGLUE_ROOT
            )
        )

        print(
            "-" * 100
        )


        for lineno, line in hits:

            print(
                f"{lineno:4d} | {line}"
            )


print(
    "\nPASS: SpatialGlue API inspection complete."
)

SPATIALGLUE SOURCE TREE
SpatialGlue/SpatialGlue_pyG.py
SpatialGlue/model.py
SpatialGlue/preprocess.py
SpatialGlue/utils.py
__init__.py
setup.py

IMPORTANT API DEFINITIONS

----------------------------------------------------------------------------------------------------
SpatialGlue/SpatialGlue_pyG.py
----------------------------------------------------------------------------------------------------
   7 | class Train_SpatialGlue:
 100 |     def train(self):

----------------------------------------------------------------------------------------------------
SpatialGlue/model.py
----------------------------------------------------------------------------------------------------
   7 | class Encoder_overall(Module):
  90 | class Encoder(Module): 

----------------------------------------------------------------------------------------------------
SpatialGlue/preprocess.py
----------------------------------------------------------------------------------------------------
  17 | def co

Cell 5：直接提取作者官方 Lymph Node / Mouse Brain notebook 的关键代码

In [5]:
# ============================================================
# Cell 5
# Extract relevant code from official reproduction notebooks
#
# We only PRINT code.
# Nothing is executed.
# ============================================================

import json
from pathlib import Path


NOTEBOOK_PATTERNS = [
    "*Lymph_Node*.ipynb",
    "*Mouse_Brain*.ipynb",
]


KEY_TERMS = [
    "SpatialGlue",
    "construct_neighbor_graph",
    "adjacent_matrix_preprocessing",
    "Train",
    "train",
    "clustering",
    "mclust",
    "kmeans",
    "ARI",
    "NMI",
    "adata",
    "h5ad",
    "RNA",
    "ATAC",
    "ADT",
    "protein",
]


selected_notebooks = []


for pattern in NOTEBOOK_PATTERNS:

    selected_notebooks.extend(
        SPATIALGLUE_NOTEBOOK_ROOT.glob(
            pattern
        )
    )


assert selected_notebooks, (
    "No official benchmark notebooks found."
)


for nb_path in selected_notebooks:

    print(
        "\n" + "=" * 120
    )

    print(
        nb_path.name
    )

    print(
        "=" * 120
    )


    with nb_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        nb = json.load(f)


    for cell_index, cell in enumerate(
        nb.get(
            "cells",
            []
        )
    ):

        if cell.get(
            "cell_type"
        ) != "code":

            continue


        source = "".join(
            cell.get(
                "source",
                []
            )
        )


        if not any(
            term.lower()
            in source.lower()
            for term in KEY_TERMS
        ):

            continue


        print(
            "\n"
            + "-" * 100
        )

        print(
            f"CODE CELL {cell_index}"
        )

        print(
            "-" * 100
        )

        print(
            source[:6000]
        )


print(
    "\n" + "=" * 120
)

print(
    "PASS: official workflow code extracted."
)

print(
    "=" * 120
)


Benchmarking_Lymph_Node_Fig1hk_FigS8cehi.ipynb

----------------------------------------------------------------------------------------------------
CODE CELL 3
----------------------------------------------------------------------------------------------------
dataset = 'Dataset11_Lymph_Node_A1'
path = '../result/'  # please replace the path with download path.
adata = sc.read_h5ad(path + dataset + '/' + 'adata_all_human_lymph_node_A1.h5ad')

----------------------------------------------------------------------------------------------------
CODE CELL 4
----------------------------------------------------------------------------------------------------
# Dataset12_Lymph_Node_D1
list_Seurat = [4,0,5,1,2,3]
adata.obs['Seurat']  = pd.Categorical(adata.obs['Seurat'], 
                      categories=list_Seurat,
                      ordered=True)
list_totalVI = [4,6,2,1,5,3]
adata.obs['totalVI']  = pd.Categorical(adata.obs['totalVI'], 
                      categories=list_totalVI,
   

Cell 6：安装 Python 3.12 可兼容环境并导入 SpatialGlue

In [7]:
# ============================================================
# Cell 6 - FIXED
# SpatialGlue compatibility setup for Kaggle / Python 3.12
#
# Fixes upstream repository packaging issue:
#   repo/
#     __init__.py          <- misplaced package init
#     SpatialGlue/
#       model.py
#       preprocess.py
#       SpatialGlue_pyG.py
#
# We create ONLY the missing package __init__.py.
# Algorithm source files are NOT modified.
# ============================================================

import sys
import subprocess
import shutil
import importlib
from pathlib import Path


# ============================================================
# 1. Install runtime dependencies
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "anndata==0.11.4",
        "scanpy==1.11.4",
        "scikit-misc",
    ],
    check=True,
)


# ============================================================
# 2. Inspect official repository layout
# ============================================================

PACKAGE_DIR = (
    SPATIALGLUE_ROOT
    / "SpatialGlue"
)

ROOT_INIT = (
    SPATIALGLUE_ROOT
    / "__init__.py"
)

PACKAGE_INIT = (
    PACKAGE_DIR
    / "__init__.py"
)


required_source_files = [
    PACKAGE_DIR / "model.py",
    PACKAGE_DIR / "preprocess.py",
    PACKAGE_DIR / "SpatialGlue_pyG.py",
    PACKAGE_DIR / "utils.py",
]


assert PACKAGE_DIR.exists(), (
    f"Missing package directory:\n{PACKAGE_DIR}"
)

assert ROOT_INIT.exists(), (
    f"Missing official root __init__.py:\n{ROOT_INIT}"
)


for path in required_source_files:
    assert path.exists(), (
        f"Missing SpatialGlue source file:\n{path}"
    )


print("=" * 100)
print("OFFICIAL SPATIALGLUE PACKAGE LAYOUT")
print("=" * 100)

print("Repository root :", SPATIALGLUE_ROOT)
print("Package dir     :", PACKAGE_DIR)
print("Root __init__   :", ROOT_INIT)
print("Package __init__:", PACKAGE_INIT)


# ============================================================
# 3. Compatibility repair
#
# Copy the official __init__.py into the actual package folder.
#
# This does NOT alter model.py / preprocess.py / training code.
# ============================================================

if not PACKAGE_INIT.exists():

    shutil.copy2(
        ROOT_INIT,
        PACKAGE_INIT,
    )

    print(
        "\nCreated missing package __init__.py:"
    )

    print(
        PACKAGE_INIT
    )

else:

    print(
        "\nPackage __init__.py already exists."
    )


# ============================================================
# 4. Make the CORRECT package location highest priority
# ============================================================

repo_path = str(
    SPATIALGLUE_ROOT
)


# remove duplicate occurrence first
sys.path = [
    p
    for p in sys.path
    if p != repo_path
]


sys.path.insert(
    0,
    repo_path,
)


# ============================================================
# 5. Remove any failed/stale SpatialGlue imports
# ============================================================

stale_modules = [
    name
    for name in list(sys.modules)
    if (
        name == "SpatialGlue"
        or
        name.startswith("SpatialGlue.")
    )
]


for name in stale_modules:

    del sys.modules[name]


importlib.invalidate_caches()


# ============================================================
# 6. Import SpatialGlue
# ============================================================

import SpatialGlue

from SpatialGlue.SpatialGlue_pyG import (
    Train_SpatialGlue,
)

from SpatialGlue.preprocess import (
    fix_seed,
    construct_neighbor_graph,
    adjacent_matrix_preprocessing,
    clr_normalize_each_cell,
    pca,
    lsi,
)


# ============================================================
# 7. Environment audit
# ============================================================

import numpy as np
import pandas as pd
import scipy
import sklearn
import anndata
import scanpy as sc
import torch


print(
    "\n" + "=" * 100
)

print(
    "SPATIALGLUE IMPORT AUDIT"
)

print(
    "=" * 100
)


print(
    "SpatialGlue loaded from:"
)

print(
    SpatialGlue.__file__
)


print(
    "\nTrain_SpatialGlue:"
)

print(
    Train_SpatialGlue
)


print(
    "\nPython        :",
    sys.version.split()[0],
)

print(
    "PyTorch       :",
    torch.__version__,
)

print(
    "NumPy         :",
    np.__version__,
)

print(
    "SciPy         :",
    scipy.__version__,
)

print(
    "scikit-learn  :",
    sklearn.__version__,
)

print(
    "AnnData       :",
    anndata.__version__,
)

print(
    "Scanpy        :",
    sc.__version__,
)

print(
    "CUDA          :",
    torch.cuda.is_available(),
)


if torch.cuda.is_available():

    print(
        "GPU           :",
        torch.cuda.get_device_name(0),
    )


# ============================================================
# 8. Critical path audit
# ============================================================

expected_init = (
    PACKAGE_INIT.resolve()
)

actual_init = Path(
    SpatialGlue.__file__
).resolve()


assert actual_init == expected_init, (
    "\nWrong SpatialGlue package imported!\n"
    f"Expected: {expected_init}\n"
    f"Actual:   {actual_init}"
)


print(
    "\nPASS: SpatialGlue imported from "
    "the correct source package."
)

print(
    "PASS: model/preprocess/training "
    "source files were NOT modified."
)

OFFICIAL SPATIALGLUE PACKAGE LAYOUT
Repository root : /kaggle/working/SpatialGlue
Package dir     : /kaggle/working/SpatialGlue/SpatialGlue
Root __init__   : /kaggle/working/SpatialGlue/__init__.py
Package __init__: /kaggle/working/SpatialGlue/SpatialGlue/__init__.py

Created missing package __init__.py:
/kaggle/working/SpatialGlue/SpatialGlue/__init__.py

SPATIALGLUE IMPORT AUDIT
SpatialGlue loaded from:
/kaggle/working/SpatialGlue/SpatialGlue/__init__.py

Train_SpatialGlue:
<class 'SpatialGlue.SpatialGlue_pyG.Train_SpatialGlue'>

Python        : 3.12.13
PyTorch       : 2.10.0+cu128
NumPy         : 2.0.2
SciPy         : 1.16.3
scikit-learn  : 1.6.1
AnnData       : 0.11.4
Scanpy        : 1.11.4
CUDA          : True
GPU           : Tesla T4

PASS: SpatialGlue imported from the correct source package.
PASS: model/preprocess/training source files were NOT modified.


Cell 7：一次性审计 5 个数据集

In [8]:
# ============================================================
# Cell 7
# Audit all five datasets before SpatialGlue preprocessing
#
# READ ONLY
# ============================================================

from pathlib import Path
import anndata as ad
import numpy as np
import scipy.sparse as sp


DATASET_SPECS = {

    "HLN-A1": {
        "omics1": (
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_RNA.h5ad"
        ),
        "omics2": (
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_ADT.h5ad"
        ),
        "omics1_name": "RNA",
        "omics2_name": "ADT",
        "datatype": "10x",
        "K": 10,
    },

    "HLN-D1": {
        "omics1": (
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_RNA.h5ad"
        ),
        "omics2": (
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_ADT.h5ad"
        ),
        "omics1_name": "RNA",
        "omics2_name": "ADT",
        "datatype": "10x",
        "K": 11,
    },

    "E18.5": {
        "omics1": (
            DATA_ROOT
            / "E18.5_mouse_brain/adata_RNA.h5ad"
        ),
        "omics2": (
            DATA_ROOT
            / "E18.5_mouse_brain/adata_ATAC.h5ad"
        ),
        "omics1_name": "RNA",
        "omics2_name": "ATAC",
        "datatype":
            "Spatial-epigenome-transcriptome",
        "K": 14,
    },

    "S2-E15": {
        "omics1": (
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_RNA.h5ad"
        ),
        "omics2": (
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_ATAC.h5ad"
        ),
        "omics1_name": "RNA",
        "omics2_name": "ATAC",
        "datatype":
            "Spatial-epigenome-transcriptome",
        "K": 15,
    },

    "S2-E18": {
        "omics1": (
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_RNA.h5ad"
        ),
        "omics2": (
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad"
        ),
        "omics1_name": "RNA",
        "omics2_name": "ATAC",
        "datatype":
            "Spatial-epigenome-transcriptome",
        "K": 16,
    },
}


LABEL_CANDIDATES = [
    "Spatial_Label",
    "spatial_label",
    "Ground Truth",
    "ground_truth",
    "label",
    "labels",
    "annotation",
]


def sampled_x_stats(
    adata,
    n_rows=128,
    n_cols=256,
):

    nr = min(
        n_rows,
        adata.n_obs,
    )

    nc = min(
        n_cols,
        adata.n_vars,
    )


    x = adata.X[
        :nr,
        :nc,
    ]


    if sp.issparse(x):

        values = x.data

    else:

        values = np.asarray(
            x
        ).ravel()


    values = values[
        np.isfinite(values)
    ]


    if len(values) == 0:

        return {
            "min": np.nan,
            "max": np.nan,
            "mean": np.nan,
            "integer_fraction": np.nan,
        }


    integer_fraction = np.mean(
        np.isclose(
            values,
            np.round(values),
            atol=1e-7,
        )
    )


    return {
        "min":
            float(values.min()),

        "max":
            float(values.max()),

        "mean":
            float(values.mean()),

        "integer_fraction":
            float(integer_fraction),
    }


for dataset, spec in (
    DATASET_SPECS.items()
):

    print(
        "\n" + "=" * 115
    )

    print(
        f"{dataset} | "
        f"{spec['omics1_name']} + "
        f"{spec['omics2_name']} | "
        f"K={spec['K']} | "
        f"datatype={spec['datatype']}"
    )

    print(
        "=" * 115
    )


    for key in [
        "omics1",
        "omics2",
    ]:

        path = spec[key]

        assert path.exists(), path


        a = ad.read_h5ad(
            path,
            backed="r",
        )


        try:

            print(
                f"\n{path.name}"
            )

            print(
                "shape       :",
                a.shape,
            )

            print(
                "X dtype     :",
                a.X.dtype,
            )

            print(
                "obs columns :",
                list(
                    a.obs.columns
                ),
            )

            print(
                "obsm keys   :",
                list(
                    a.obsm.keys()
                ),
            )

            print(
                "layers      :",
                list(
                    a.layers.keys()
                ),
            )


            if "spatial" in a.obsm:

                spatial = np.asarray(
                    a.obsm["spatial"]
                )

                print(
                    "spatial     :",
                    spatial.shape,
                )

                print(
                    "spatial finite:",
                    bool(
                        np.isfinite(
                            spatial
                        ).all()
                    ),
                )

            else:

                print(
                    "spatial     : MISSING"
                )


            stats = sampled_x_stats(
                a
            )

            print(
                "X sample    :",
                stats,
            )


            found_labels = [
                x
                for x
                in LABEL_CANDIDATES
                if x in a.obs.columns
            ]


            for label_key in found_labels:

                y = (
                    a.obs[
                        label_key
                    ]
                    .dropna()
                )

                print(
                    f"label {label_key!r}: "
                    f"n={len(y)}, "
                    f"unique={y.nunique()}"
                )


        finally:

            a.file.close()


    # --------------------------------------------------------
    # Pair alignment audit
    # --------------------------------------------------------

    a1 = ad.read_h5ad(
        spec["omics1"],
        backed="r",
    )

    a2 = ad.read_h5ad(
        spec["omics2"],
        backed="r",
    )


    try:

        same_names = np.array_equal(
            np.asarray(
                a1.obs_names
            ),
            np.asarray(
                a2.obs_names
            ),
        )


        print(
            "\nSpot names aligned:",
            same_names,
        )


        if (
            "spatial" in a1.obsm
            and
            "spatial" in a2.obsm
        ):

            s1 = np.asarray(
                a1.obsm["spatial"]
            )

            s2 = np.asarray(
                a2.obsm["spatial"]
            )


            same_spatial = (
                s1.shape == s2.shape
                and
                np.allclose(
                    s1,
                    s2,
                )
            )


            print(
                "Spatial coordinates aligned:",
                same_spatial,
            )


    finally:

        a1.file.close()
        a2.file.close()


print(
    "\n" + "=" * 115
)

print(
    "PASS: five-dataset metadata audit complete."
)


HLN-A1 | RNA + ADT | K=10 | datatype=10x

adata_RNA.h5ad
shape       : (3484, 18085)
X dtype     : float32
obs columns : ['Spatial_Label']
obsm keys   : ['spatial']
layers      : []
spatial     : (3484, 2)
spatial finite: True


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


X sample    : {'min': 1.0, 'max': 19.0, 'mean': 1.2986036539077759, 'integer_fraction': 1.0}
label 'Spatial_Label': n=3484, unique=10

adata_ADT.h5ad
shape       : (3484, 31)
X dtype     : float32
obs columns : ['Spatial_Label']
obsm keys   : ['spatial']
layers      : []
spatial     : (3484, 2)
spatial finite: True
X sample    : {'min': 58.0, 'max': 470000.0, 'mean': 15649.0771484375, 'integer_fraction': 1.0}
label 'Spatial_Label': n=3484, unique=10


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



Spot names aligned: True
Spatial coordinates aligned: True

HLN-D1 | RNA + ADT | K=11 | datatype=10x

adata_RNA.h5ad
shape       : (3359, 18085)
X dtype     : float32
obs columns : ['Spatial_Label']
obsm keys   : ['spatial']
layers      : []
spatial     : (3359, 2)
spatial finite: True
X sample    : {'min': 1.0, 'max': 6.0, 'mean': 1.1559139490127563, 'integer_fraction': 1.0}
label 'Spatial_Label': n=3359, unique=11


...


Spot names aligned: True
Spatial coordinates aligned: True

S2-E18 | RNA + ATAC | K=16 | datatype=Spatial-epigenome-transcriptome


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



adata_RNA.h5ad
shape       : (2248, 32285)
X dtype     : float32
obs columns : ['Spatial_Label']
obsm keys   : ['ATAC_Clusters', 'Combined_Clusters', 'RNA_Clusters', 'spatial']
layers      : []
spatial     : (2248, 2)
spatial finite: True
X sample    : {'min': 1.0, 'max': 26.0, 'mean': 1.8801802396774292, 'integer_fraction': 1.0}
label 'Spatial_Label': n=2248, unique=16

adata_ATAC.h5ad
shape       : (2248, 94941)
X dtype     : float32
obs columns : ['Spatial_Label']
obsm keys   : ['ATAC_Clusters', 'Combined_Clusters', 'RNA_Clusters', 'spatial']
layers      : []
spatial     : (2248, 2)
spatial finite: True
X sample    : {'min': 1.0, 'max': 12.0, 'mean': 2.407867431640625, 'integer_fraction': 1.0}
label 'Spatial_Label': n=2248, unique=16


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



Spot names aligned: True
Spatial coordinates aligned: True

PASS: five-dataset metadata audit complete.


Cell 7.5：确认 E18.5 的坐标和 GT 标签

In [9]:
# ============================================================
# Cell 7.5
# Audit E18.5 coordinate / ground-truth candidates
#
# READ ONLY
# ============================================================

import anndata as ad
import numpy as np
import pandas as pd


E185_RNA_PATH = (
    DATA_ROOT
    / "E18.5_mouse_brain"
    / "adata_RNA.h5ad"
)

E185_ATAC_PATH = (
    DATA_ROOT
    / "E18.5_mouse_brain"
    / "adata_ATAC.h5ad"
)


rna = ad.read_h5ad(
    E185_RNA_PATH,
    backed="r",
)

atac = ad.read_h5ad(
    E185_ATAC_PATH,
    backed="r",
)


print("=" * 110)
print("E18.5 COORDINATE / LABEL AUDIT")
print("=" * 110)


# ============================================================
# 1. Alignment
# ============================================================

print(
    "Spot names aligned:",
    np.array_equal(
        np.asarray(rna.obs_names),
        np.asarray(atac.obs_names),
    ),
)


# ============================================================
# 2. Coordinates from array_col / array_row
# ============================================================

for col in [
    "array_col",
    "array_row",
]:

    assert col in rna.obs.columns
    assert col in atac.obs.columns


rna_xy = np.column_stack(
    [
        np.asarray(
            rna.obs["array_col"],
            dtype=float,
        ),
        np.asarray(
            rna.obs["array_row"],
            dtype=float,
        ),
    ]
)

atac_xy = np.column_stack(
    [
        np.asarray(
            atac.obs["array_col"],
            dtype=float,
        ),
        np.asarray(
            atac.obs["array_row"],
            dtype=float,
        ),
    ]
)


print(
    "\nCoordinate shape:",
    rna_xy.shape,
)

print(
    "RNA coordinates finite:",
    np.isfinite(
        rna_xy
    ).all(),
)

print(
    "ATAC coordinates finite:",
    np.isfinite(
        atac_xy
    ).all(),
)

print(
    "RNA/ATAC coordinates aligned:",
    np.allclose(
        rna_xy,
        atac_xy,
    ),
)

print(
    "Unique coordinate rows:",
    np.unique(
        rna_xy,
        axis=0,
    ).shape[0],
)


print(
    "\nFirst 5 coordinates:"
)

print(
    rna_xy[:5]
)


# ============================================================
# 3. Candidate ground-truth columns
# ============================================================

candidate_columns = [
    "ATAC_Clusters",
    "RNA_Clusters",
    "Combined_Clusters",
    "Combined_Clusters_annotation",
    "src",
]


print(
    "\n" + "=" * 110
)

print(
    "GROUND-TRUTH CANDIDATES"
)

print(
    "=" * 110
)


for col in candidate_columns:

    if col not in rna.obs.columns:
        continue

    rna_values = (
        rna.obs[col]
        .astype(str)
    )

    atac_values = (
        atac.obs[col]
        .astype(str)
    )


    print(
        f"\nColumn: {col}"
    )

    print(
        "  RNA unique :",
        rna_values.nunique(),
    )

    print(
        "  ATAC unique:",
        atac_values.nunique(),
    )

    print(
        "  Missing RNA:",
        int(
            rna.obs[col]
            .isna()
            .sum()
        ),
    )

    print(
        "  RNA/ATAC identical:",
        np.array_equal(
            rna_values.to_numpy(),
            atac_values.to_numpy(),
        ),
    )


    counts = (
        rna.obs[col]
        .value_counts(
            dropna=False
        )
    )

    print(
        "  Value counts:"
    )

    print(
        counts.to_string()
    )


# ============================================================
# 4. Specifically test expected K=14
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "EXPECTED E18.5 K = 14"
)

print(
    "=" * 110
)


for col in candidate_columns:

    if col not in rna.obs.columns:
        continue

    n_unique = (
        rna.obs[col]
        .dropna()
        .nunique()
    )

    if n_unique == 14:

        print(
            f"14-cluster candidate: {col}"
        )


rna.file.close()
atac.file.close()


print(
    "\nPASS: E18.5 metadata audit complete."
)

E18.5 COORDINATE / LABEL AUDIT
Spot names aligned: True

Coordinate shape: (2129, 2)
RNA coordinates finite: True
ATAC coordinates finite: True
RNA/ATAC coordinates aligned: True
Unique coordinate rows: 2129

First 5 coordinates:
[[43. 15.]
 [46. 23.]
 [16. 35.]
 [17. 36.]
 [12.  5.]]

GROUND-TRUTH CANDIDATES

Column: ATAC_Clusters
  RNA unique : 10
  ATAC unique: 10
  Missing RNA: 0
  RNA/ATAC identical: True
  Value counts:
ATAC_Clusters
4     590
10    533
13    332
9     242
12    167
8     160
2      57
1      31
3      11
7       6

Column: RNA_Clusters
  RNA unique : 13
  ATAC unique: 13
  Missing RNA: 0
  RNA/ATAC identical: True
  Value counts:
RNA_Clusters
4     597
5     343
13    332
11    283
3     188
8     156
2     150
6      31
9      16
10     15
7      11
12      6
1       1

Column: Combined_Clusters
  RNA unique : 14
  ATAC unique: 14
  Missing RNA: 0
  RNA/ATAC identical: True
  Value counts:
Combined_Clusters
13    671
8     275
11    239
6     236
7     178
15  

Cell 8：检查官方 mclust 在 Kaggle 能不能直接用

In [10]:
# ============================================================
# Cell 8
# Check R + mclust availability
#
# READ ONLY
# ============================================================

import shutil
import subprocess


print("=" * 100)
print("R / MCLUST AUDIT")
print("=" * 100)


r_path = shutil.which(
    "R"
)

rscript_path = shutil.which(
    "Rscript"
)


print(
    "R      :",
    r_path,
)

print(
    "Rscript:",
    rscript_path,
)


if rscript_path is None:

    print(
        "\nMCLUST_STATUS = R_NOT_AVAILABLE"
    )

else:

    proc = subprocess.run(
        [
            rscript_path,
            "-e",
            (
                'if (requireNamespace("mclust", '
                'quietly=TRUE)) {'
                'cat("MCLUST_INSTALLED\\n");'
                'cat(as.character('
                'packageVersion("mclust")))'
                '} else {'
                'cat("MCLUST_NOT_INSTALLED")'
                '}'
            ),
        ],
        text=True,
        capture_output=True,
    )


    print(
        "\nstdout:"
    )

    print(
        proc.stdout
    )

    print(
        "stderr:"
    )

    print(
        proc.stderr
    )

    print(
        "return code:",
        proc.returncode,
    )


print(
    "\nPASS: clustering-environment audit complete."
)

R / MCLUST AUDIT
R      : /usr/local/bin/R
Rscript: /usr/bin/Rscript

stdout:
MCLUST_NOT_INSTALLED
stderr:

return code: 0

PASS: clustering-environment audit complete.


Cell 9：安装并验证 R mclust

In [11]:
# ============================================================
# Cell 9
# Install and verify R package mclust
# ============================================================

import os
import subprocess
import shutil


assert shutil.which("R") is not None
assert shutil.which("Rscript") is not None


# ------------------------------------------------------------
# Detect R_HOME automatically
# ------------------------------------------------------------

R_HOME = subprocess.check_output(
    [
        "R",
        "RHOME",
    ],
    text=True,
).strip()


os.environ["R_HOME"] = R_HOME


print("=" * 100)
print("R / MCLUST SETUP")
print("=" * 100)

print(
    "R_HOME =",
    R_HOME,
)


# ------------------------------------------------------------
# Check whether mclust already exists
# ------------------------------------------------------------

check = subprocess.run(
    [
        "Rscript",
        "-e",
        (
            'cat(ifelse('
            'requireNamespace("mclust", quietly=TRUE),'
            '"YES","NO"))'
        ),
    ],
    text=True,
    capture_output=True,
    check=True,
)


installed = (
    check.stdout.strip()
    == "YES"
)


print(
    "mclust installed:",
    installed,
)


# ------------------------------------------------------------
# Install if missing
# ------------------------------------------------------------

if not installed:

    print(
        "\nInstalling R package mclust..."
    )

    subprocess.run(
        [
            "Rscript",
            "-e",
            (
                'install.packages('
                '"mclust", '
                'repos="https://cloud.r-project.org")'
            ),
        ],
        check=True,
    )


# ------------------------------------------------------------
# Final R-side audit
# ------------------------------------------------------------

verify = subprocess.run(
    [
        "Rscript",
        "-e",
        (
            'library(mclust); '
            'cat("MCLUST_VERSION=", '
            'as.character(packageVersion("mclust")), '
            '"\\n", sep="")'
        ),
    ],
    text=True,
    capture_output=True,
    check=True,
)


print(
    "\nR verification:"
)

print(
    verify.stdout.strip()
)


# ------------------------------------------------------------
# Python/rpy2-side audit
# ------------------------------------------------------------

from rpy2.robjects.packages import (
    importr,
)


mclust_pkg = importr(
    "mclust"
)


print(
    "\nPASS: R mclust is accessible "
    "from Python/rpy2."
)

R / MCLUST SETUP
R_HOME = /usr/lib/R
mclust installed: False

Installing R package mclust...


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/mclust_6.1.3.tar.gz'
Content type 'application/x-gzip' length 2780882 bytes (2.7 MB)
downloaded 2.7 MB

* installing *source* package ‘mclust’ ...
** this is package ‘mclust’ version ‘6.1.3’
** package ‘mclust’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C compiler: ‘cc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0’
using Fortran compiler: ‘GNU Fortran (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0’


cc -I"/usr/share/R/include" -DNDEBUG       -fpic  -g -O2 -ffile-prefix-map=/build/r-base-oO8drT/r-base-4.6.0=. -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2  -c init.c -o init.o
f77  -fpic  -g -O2 -ffile-prefix-map=/build/r-base-oO8drT/r-base-4.6.0=. -fstack-protector-strong  -c mclust.f -o mclust.o
f77  -fpic  -g -O2 -ffile-prefix-map=/build/r-base-oO8drT/r-base-4.6.0=. -fstack-protector-strong  -c mclust2.f -o mclust2.o
f77  -fpic  -g -O2 -ffile-prefix-map=/build/r-base-oO8drT/r-base-4.6.0=. -fstack-protector-strong  -c util.f -o util.o
cc -shared -L/usr/lib/R/lib -Wl,-Bsymbolic-functions -flto=auto -ffat-lto-objects -flto=auto -Wl,-z,relro -o mclust.so init.o mclust.o mclust2.o util.o -llapack -lblas -lgfortran -lm -lquadmath -lgfortran -lm -lquadmath -L/usr/lib/R/lib -lR


installing to /usr/local/lib/R/site-library/00LOCK-mclust/00new/mclust/libs
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (mclust)

The downloaded source packages are in
	‘/tmp/RtmpIGbngN/downloaded_packages’



R verification:
MCLUST_VERSION=6.1.3

PASS: R mclust is accessible from Python/rpy2.


Cell 10：建立统一 SpatialGlue 数据适配器

In [21]:
# ============================================================
# Cell 10 - FINAL
# Unified SpatialGlue dataset adapter
#
# Supported datasets:
#   HLN-A1   : RNA + ADT
#   HLN-D1   : RNA + ADT
#   E18.5    : RNA + ATAC
#   S2-E15   : RNA + ATAC
#   S2-E18   : RNA + ATAC
#
# Important benchmark rule:
#   ALL methods must be evaluated on the SAME spots.
#
# Therefore:
#   - feature filtering is allowed
#   - cell/spot filtering is NOT allowed
#
# Also includes:
#   - modern SciPy-compatible CLR normalization
#   - E18.5 coordinate / GT metadata adapter
# ============================================================

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

from SpatialGlue.preprocess import (
    pca,
    lsi,
    construct_neighbor_graph,
    fix_seed,
)


# ============================================================
# 0. SciPy-compatible SpatialGlue CLR
#
# Same mathematical formula as official SpatialGlue.
# Only replaces sparse_matrix.A with .toarray().
# ============================================================

def clr_normalize_each_cell_compat(
    adata,
    inplace=True,
):

    if not inplace:
        adata = adata.copy()


    # --------------------------------------------------------
    # Convert sparse matrix to dense
    # --------------------------------------------------------

    if sp.issparse(
        adata.X
    ):

        X = adata.X.toarray()

    else:

        X = np.asarray(
            adata.X
        )


    X = np.asarray(
        X,
        dtype=np.float64,
    )


    # --------------------------------------------------------
    # Same Seurat CLR formula used by SpatialGlue
    # --------------------------------------------------------

    def seurat_clr(x):

        positive = (
            x > 0
        )

        s = np.sum(
            np.log1p(
                x[positive]
            )
        )

        exp_value = np.exp(
            s / len(x)
        )

        return np.log1p(
            x / exp_value
        )


    adata.X = np.apply_along_axis(
        seurat_clr,
        1,
        X,
    )


    return adata


# ============================================================
# 1. Unified dataset preparation
# ============================================================

def prepare_spatialglue_dataset(
    dataset_name,
):

    # --------------------------------------------------------
    # Dataset audit
    # --------------------------------------------------------

    assert (
        dataset_name
        in DATASET_SPECS
    ), (
        f"Unknown dataset: {dataset_name}"
    )


    spec = DATASET_SPECS[
        dataset_name
    ]


    print(
        "\n" + "=" * 110
    )

    print(
        f"PREPARING SPATIALGLUE DATASET: "
        f"{dataset_name}"
    )

    print(
        "=" * 110
    )


    # ========================================================
    # 2. Load two modalities
    # ========================================================

    adata_omics1 = sc.read_h5ad(
        spec["omics1"]
    )

    adata_omics2 = sc.read_h5ad(
        spec["omics2"]
    )


    # Some source files contain duplicated var names.
    # Make them unique on the in-memory copy only.
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()


    # --------------------------------------------------------
    # Spot alignment must already match.
    # --------------------------------------------------------

    assert np.array_equal(
        np.asarray(
            adata_omics1.obs_names
        ),
        np.asarray(
            adata_omics2.obs_names
        ),
    ), (
        f"{dataset_name}: "
        "omics1 / omics2 spot names are not aligned."
    )


    n_before = int(
        adata_omics1.n_obs
    )


    assert (
        adata_omics2.n_obs
        == n_before
    )


    print(
        "Initial spots:",
        n_before,
    )


    # ========================================================
    # 3. E18.5 metadata adapter
    #
    # Original files contain:
    #   array_col
    #   array_row
    #   Combined_Clusters_annotation
    #
    # but do not contain:
    #   obsm["spatial"]
    #   obs["Spatial_Label"]
    # ========================================================

    if dataset_name == "E18.5":

        # ----------------------------------------------------
        # Coordinates
        # ----------------------------------------------------

        coordinates = np.column_stack(
            [
                np.asarray(
                    adata_omics1.obs[
                        "array_col"
                    ],
                    dtype=np.float32,
                ),

                np.asarray(
                    adata_omics1.obs[
                        "array_row"
                    ],
                    dtype=np.float32,
                ),
            ]
        )


        assert (
            coordinates.shape
            == (
                n_before,
                2,
            )
        )

        assert np.isfinite(
            coordinates
        ).all()


        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        labels = (
            adata_omics1.obs[
                "Combined_Clusters_annotation"
            ]
            .astype(str)
            .to_numpy()
        )


        assert (
            len(
                np.unique(labels)
            )
            == spec["K"]
        )


        # ----------------------------------------------------
        # Copy identical metadata to both modalities
        # ----------------------------------------------------

        for adata in [
            adata_omics1,
            adata_omics2,
        ]:

            adata.obsm[
                "spatial"
            ] = (
                coordinates.copy()
            )

            adata.obs[
                "Spatial_Label"
            ] = (
                labels.copy()
            )


        print(
            "E18.5 metadata adapter: PASS"
        )


    # ========================================================
    # 4. Common metadata audit
    # ========================================================

    for adata in [
        adata_omics1,
        adata_omics2,
    ]:

        assert (
            "Spatial_Label"
            in adata.obs.columns
        ), (
            f"{dataset_name}: "
            "Spatial_Label missing."
        )


        assert (
            "spatial"
            in adata.obsm
        ), (
            f"{dataset_name}: "
            "spatial coordinates missing."
        )


        coordinates = np.asarray(
            adata.obsm[
                "spatial"
            ]
        )


        assert (
            coordinates.shape
            == (
                n_before,
                2,
            )
        )


        assert np.isfinite(
            coordinates
        ).all()


    # --------------------------------------------------------
    # Modalities must share exact coordinates
    # --------------------------------------------------------

    assert np.allclose(
        np.asarray(
            adata_omics1.obsm[
                "spatial"
            ]
        ),
        np.asarray(
            adata_omics2.obsm[
                "spatial"
            ]
        ),
    ), (
        f"{dataset_name}: "
        "spatial coordinates do not match."
    )


    # --------------------------------------------------------
    # Ground-truth cluster number
    # --------------------------------------------------------

    labels = (
        adata_omics1.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    assert (
        len(
            np.unique(labels)
        )
        == spec["K"]
    ), (
        f"{dataset_name}: "
        f"expected K={spec['K']}, "
        f"got "
        f"{len(np.unique(labels))}"
    )


    # ========================================================
    # 5A. RNA + ADT
    #
    # HLN-A1 / HLN-D1
    # datatype = 10x
    # ========================================================

    if (
        spec["datatype"]
        == "10x"
    ):

        print(
            "\nPreprocessing mode: "
            "RNA + ADT / 10x"
        )


        # ====================================================
        # RNA
        # ====================================================

        # Feature filtering only.
        # Does NOT change spots.
        sc.pp.filter_genes(
            adata_omics1,
            min_cells=10,
        )


        assert (
            adata_omics1.n_obs
            == n_before
        )


        # ----------------------------------------------------
        # HVG
        # ----------------------------------------------------

        sc.pp.highly_variable_genes(
            adata_omics1,
            flavor="seurat_v3",
            n_top_genes=3000,
        )


        # ----------------------------------------------------
        # Normalize + log
        # ----------------------------------------------------

        sc.pp.normalize_total(
            adata_omics1,
            target_sum=1e4,
        )


        sc.pp.log1p(
            adata_omics1
        )


        # ----------------------------------------------------
        # Scale
        # ----------------------------------------------------

        sc.pp.scale(
            adata_omics1
        )


        # ----------------------------------------------------
        # Use only HVGs for PCA
        # ----------------------------------------------------

        adata_rna_high = (
            adata_omics1[
                :,
                adata_omics1.var[
                    "highly_variable"
                ],
            ].copy()
        )


        # ----------------------------------------------------
        # Official lymph-node setup:
        #
        # number of PCA components =
        # number of proteins - 1
        #
        # ADT has 31 features -> PCA30
        # ----------------------------------------------------

        n_components = int(
            adata_omics2.n_vars
            - 1
        )


        adata_omics1.obsm[
            "feat"
        ] = pca(
            adata_rna_high,
            n_comps=n_components,
        )


        # ====================================================
        # ADT
        # ====================================================

        adata_omics2 = (
            clr_normalize_each_cell_compat(
                adata_omics2,
                inplace=False,
            )
        )


        sc.pp.scale(
            adata_omics2
        )


        adata_omics2.obsm[
            "feat"
        ] = pca(
            adata_omics2,
            n_comps=n_components,
        )


        # ----------------------------------------------------
        # Expected feature dimensions
        # ----------------------------------------------------

        expected_dim = (
            n_components
        )


    # ========================================================
    # 5B. RNA + ATAC
    #
    # E18.5 / S2-E15 / S2-E18
    #
    # IMPORTANT:
    # We DO NOT execute:
    #
    #   sc.pp.filter_cells(
    #       adata_omics1,
    #       min_genes=200
    #   )
    #
    # because all benchmark methods must use the exact
    # same evaluation population.
    #
    # For E18.5 this preserves:
    #   2129 -> 2129
    # instead of:
    #   2129 -> 2091
    # ========================================================

    elif (
        spec["datatype"]
        == "Spatial-epigenome-transcriptome"
    ):

        print(
            "\nPreprocessing mode: "
            "RNA + ATAC / "
            "Spatial-epigenome-transcriptome"
        )


        # ====================================================
        # RNA
        # ====================================================

        # ----------------------------------------------------
        # Gene filtering only.
        #
        # This changes features,
        # NOT evaluation spots.
        # ----------------------------------------------------

        sc.pp.filter_genes(
            adata_omics1,
            min_cells=10,
        )


        # ----------------------------------------------------
        # Frozen benchmark population
        # ----------------------------------------------------

        assert (
            adata_omics1.n_obs
            == n_before
        ), (
            f"{dataset_name}: "
            f"RNA spot count changed "
            f"{n_before} -> "
            f"{adata_omics1.n_obs}"
        )


        assert (
            adata_omics2.n_obs
            == n_before
        ), (
            f"{dataset_name}: "
            f"ATAC spot count changed."
        )


        assert np.array_equal(
            np.asarray(
                adata_omics1.obs_names
            ),
            np.asarray(
                adata_omics2.obs_names
            ),
        ), (
            f"{dataset_name}: "
            "RNA / ATAC spot order mismatch."
        )


        # ----------------------------------------------------
        # RNA HVG
        # ----------------------------------------------------

        sc.pp.highly_variable_genes(
            adata_omics1,
            flavor="seurat_v3",
            n_top_genes=3000,
        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        sc.pp.normalize_total(
            adata_omics1,
            target_sum=1e4,
        )


        sc.pp.log1p(
            adata_omics1
        )


        # ----------------------------------------------------
        # Scale
        # ----------------------------------------------------

        sc.pp.scale(
            adata_omics1
        )


        # ----------------------------------------------------
        # HVG subset
        # ----------------------------------------------------

        adata_rna_high = (
            adata_omics1[
                :,
                adata_omics1.var[
                    "highly_variable"
                ],
            ].copy()
        )


        # ----------------------------------------------------
        # RNA PCA50
        # ----------------------------------------------------

        adata_omics1.obsm[
            "feat"
        ] = pca(
            adata_rna_high,
            n_comps=50,
        )


        # ====================================================
        # ATAC
        # ====================================================

        # ----------------------------------------------------
        # Compute LSI only if absent.
        # ----------------------------------------------------

        if (
            "X_lsi"
            not in adata_omics2.obsm
        ):

            # Keep this step consistent with
            # the SpatialGlue preprocessing workflow.
            sc.pp.highly_variable_genes(
                adata_omics2,
                flavor="seurat_v3",
                n_top_genes=3000,
            )


            # SpatialGlue lsi():
            # n_components=51
            # then removes the first component
            #
            # final dimensionality = 50
            lsi(
                adata_omics2,
                use_highly_variable=False,
                n_components=51,
            )


        adata_omics2.obsm[
            "feat"
        ] = np.asarray(
            adata_omics2.obsm[
                "X_lsi"
            ]
        ).copy()


        expected_dim = 50


        # ----------------------------------------------------
        # Frozen population audit again
        # ----------------------------------------------------

        assert (
            adata_omics1.n_obs
            == n_before
        )


        assert (
            adata_omics2.n_obs
            == n_before
        )


        assert np.array_equal(
            np.asarray(
                adata_omics1.obs_names
            ),
            np.asarray(
                adata_omics2.obs_names
            ),
        )


    else:

        raise ValueError(
            "Unsupported datatype: "
            f"{spec['datatype']}"
        )


    # ========================================================
    # 6. Feature representation audit
    # ========================================================

    feat1 = np.asarray(
        adata_omics1.obsm[
            "feat"
        ]
    )


    feat2 = np.asarray(
        adata_omics2.obsm[
            "feat"
        ]
    )


    print(
        "\nFeature audit:"
    )

    print(
        f"  {spec['omics1_name']} feat:",
        feat1.shape,
    )

    print(
        f"  {spec['omics2_name']} feat:",
        feat2.shape,
    )


    assert (
        feat1.shape[0]
        == n_before
    )


    assert (
        feat2.shape[0]
        == n_before
    )


    assert (
        feat1.shape[1]
        == expected_dim
    ), (
        f"{dataset_name}: "
        f"omics1 expected dim "
        f"{expected_dim}, "
        f"got {feat1.shape[1]}"
    )


    assert (
        feat2.shape[1]
        == expected_dim
    ), (
        f"{dataset_name}: "
        f"omics2 expected dim "
        f"{expected_dim}, "
        f"got {feat2.shape[1]}"
    )


    assert np.isfinite(
        feat1
    ).all()


    assert np.isfinite(
        feat2
    ).all()


    # ========================================================
    # 7. Final frozen population audit
    # ========================================================

    assert (
        adata_omics1.n_obs
        == n_before
    )


    assert (
        adata_omics2.n_obs
        == n_before
    )


    assert np.array_equal(
        np.asarray(
            adata_omics1.obs_names
        ),
        np.asarray(
            adata_omics2.obs_names
        ),
    )


    labels = (
        adata_omics1.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    assert (
        len(labels)
        == n_before
    )


    assert (
        len(
            np.unique(labels)
        )
        == spec["K"]
    )


    print(
        f"\n[{dataset_name}] "
        f"spots preserved: "
        f"{adata_omics1.n_obs}/{n_before}"
    )


    print(
        f"[{dataset_name}] "
        f"GT clusters: "
        f"{len(np.unique(labels))}"
    )


    # ========================================================
    # 8. Construct official SpatialGlue graphs
    # ========================================================

    print(
        "\nConstructing SpatialGlue graphs..."
    )


    data = construct_neighbor_graph(
        adata_omics1,
        adata_omics2,
        datatype=spec[
            "datatype"
        ],
    )


    # ========================================================
    # 9. Final report
    # ========================================================

    print(
        "\n" + "-" * 110
    )

    print(
        f"{dataset_name} PREPARATION COMPLETE"
    )

    print(
        "-" * 110
    )


    print(
        "Spots:",
        adata_omics1.n_obs,
    )


    print(
        "K:",
        spec["K"],
    )


    print(
        f"{spec['omics1_name']} feat:",
        adata_omics1.obsm[
            "feat"
        ].shape,
    )


    print(
        f"{spec['omics2_name']} feat:",
        adata_omics2.obsm[
            "feat"
        ].shape,
    )


    print(
        "Spatial:",
        np.asarray(
            adata_omics1.obsm[
                "spatial"
            ]
        ).shape,
    )


    print(
        "\nPASS: SpatialGlue dataset "
        "preparation verified."
    )


    # ========================================================
    # 10. Return
    # ========================================================

    return {

        "dataset":
            dataset_name,

        "spec":
            spec,

        "adata_omics1":
            adata_omics1,

        "adata_omics2":
            adata_omics2,

        "labels":
            labels,

        "data":
            data,

        "n_spots":
            n_before,
    }


print("=" * 110)
print("SPATIALGLUE FINAL DATASET ADAPTER")
print("=" * 110)

print(
    "PASS: prepare_spatialglue_dataset() "
    "defined."
)

print(
    "PASS: SciPy-compatible CLR integrated."
)

print(
    "PASS: benchmark spots are frozen."
)

SPATIALGLUE FINAL DATASET ADAPTER
PASS: prepare_spatialglue_dataset() defined.
PASS: SciPy-compatible CLR integrated.
PASS: benchmark spots are frozen.


Cell 10.5：兼容当前 SciPy 的 CLR

In [14]:
# ============================================================
# Cell 10.5
# SpatialGlue CLR compatibility fix for modern SciPy
#
# Official SpatialGlue uses:
#     sparse_matrix.A
#
# Modern scipy csr_matrix may not expose .A.
#
# This reproduces the SAME official CLR formula,
# but uses .toarray().
#
# SpatialGlue source files are NOT modified.
# ============================================================

import numpy as np
import scipy.sparse as sp


def clr_normalize_each_cell(
    adata,
    inplace=True,
):
    """
    Compatibility-equivalent implementation of
    SpatialGlue.preprocess.clr_normalize_each_cell.

    Same Seurat CLR formula as official SpatialGlue.
    Only sparse -> dense conversion is changed:
        .A  ->  .toarray()
    """

    if not inplace:
        adata = adata.copy()


    if sp.issparse(
        adata.X
    ):

        X = adata.X.toarray()

    else:

        X = np.asarray(
            adata.X
        )


    # ADT matrix is small; float64 keeps
    # the behavior numerically stable.
    X = np.asarray(
        X,
        dtype=np.float64,
    )


    def seurat_clr(x):

        positive = (
            x > 0
        )

        s = np.sum(
            np.log1p(
                x[positive]
            )
        )

        exp_value = np.exp(
            s / len(x)
        )

        return np.log1p(
            x / exp_value
        )


    adata.X = np.apply_along_axis(
        seurat_clr,
        1,
        X,
    )


    return adata


print("=" * 100)
print("SPATIALGLUE CLR COMPATIBILITY PATCH")
print("=" * 100)

print(
    "PASS: notebook-level CLR function replaced."
)

print(
    "Official SpatialGlue source files were NOT modified."
)

SPATIALGLUE CLR COMPATIBILITY PATCH
PASS: notebook-level CLR function replaced.
Official SpatialGlue source files were NOT modified.


Cell 11：只预处理 HLN-A1，不训练

In [15]:
# ============================================================
# Cell 11
# HLN-A1 preprocessing smoke test
#
# NO MODEL TRAINING
# ============================================================

HLNA1_PREP = (
    prepare_spatialglue_dataset(
        "HLN-A1"
    )
)


a1 = HLNA1_PREP[
    "adata_omics1"
]

a2 = HLNA1_PREP[
    "adata_omics2"
]


print("=" * 100)
print("HLN-A1 SPATIALGLUE PREPROCESSING")
print("=" * 100)


print(
    "Spots:",
    a1.n_obs,
)

print(
    "GT clusters:",
    len(
        np.unique(
            HLNA1_PREP[
                "labels"
            ]
        )
    ),
)


print(
    "\nRNA feat:",
    a1.obsm[
        "feat"
    ].shape,
)

print(
    "ADT feat:",
    a2.obsm[
        "feat"
    ].shape,
)


print(
    "\nRNA feat finite:",
    np.isfinite(
        a1.obsm["feat"]
    ).all(),
)

print(
    "ADT feat finite:",
    np.isfinite(
        a2.obsm["feat"]
    ).all(),
)


print(
    "\nRNA spatial:",
    a1.obsm[
        "spatial"
    ].shape,
)

print(
    "ADT spatial:",
    a2.obsm[
        "spatial"
    ].shape,
)


print(
    "\nRNA spatial graph edges:",
    len(
        a1.uns[
            "adj_spatial"
        ]
    ),
)

print(
    "ADT spatial graph edges:",
    len(
        a2.uns[
            "adj_spatial"
        ]
    ),
)


print(
    "RNA feature graph:",
    a1.obsm[
        "adj_feature"
    ].shape,
)

print(
    "ADT feature graph:",
    a2.obsm[
        "adj_feature"
    ].shape,
)


assert (
    a1.n_obs
    == 3484
)

assert (
    a2.n_obs
    == 3484
)

assert (
    a1.obsm[
        "feat"
    ].shape
    == (3484, 30)
)

assert (
    a2.obsm[
        "feat"
    ].shape
    == (3484, 30)
)


print(
    "\nPASS: HLN-A1 SpatialGlue "
    "preprocessing verified."
)

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


HLN-A1 SPATIALGLUE PREPROCESSING
Spots: 3484
GT clusters: 10

RNA feat: (3484, 30)
ADT feat: (3484, 30)

RNA feat finite: True
ADT feat finite: True

RNA spatial: (3484, 2)
ADT spatial: (3484, 2)

RNA spatial graph edges: 10452
ADT spatial graph edges: 10452
RNA feature graph: (3484, 3484)
ADT feature graph: (3484, 3484)

PASS: HLN-A1 SpatialGlue preprocessing verified.


Cell 12：HLN-A1 seed0，只训练并保存 embedding

In [17]:
# ============================================================
# Cell 12
# SpatialGlue HLN-A1 seed0
#
# TRAINING ONLY
#
# Official SpatialGlue settings:
#   datatype = 10x
#   epochs   = 200 (set internally)
#
# This cell DOES NOT run mclust.
# ============================================================

from pathlib import Path
import json

import numpy as np
import torch

from SpatialGlue.SpatialGlue_pyG import (
    Train_SpatialGlue,
)

from SpatialGlue.preprocess import (
    fix_seed,
)


# ============================================================
# 1. Frozen smoke-test protocol
# ============================================================

DATASET = "HLN-A1"
TRAIN_SEED = 0
N_CLUSTERS = 10

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


assert DEVICE.type == "cuda", (
    "GPU is not enabled."
)


OUT_DIR = (
    Path("/kaggle/working")
    / "SpatialGlue_baseline"
    / "smoke"
    / "HLNA1_seed0"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Clean preprocessing
# ============================================================

fix_seed(
    TRAIN_SEED
)


prepared = (
    prepare_spatialglue_dataset(
        DATASET
    )
)


adata_omics1 = (
    prepared["adata_omics1"]
)

gt = np.asarray(
    prepared["labels"]
)

data = prepared["data"]


coords = np.asarray(
    adata_omics1.obsm["spatial"]
)


assert len(gt) == 3484
assert coords.shape == (3484, 2)
assert len(np.unique(gt)) == N_CLUSTERS


# ============================================================
# 3. Train SpatialGlue
# ============================================================

fix_seed(
    TRAIN_SEED
)


model = Train_SpatialGlue(
    data,
    datatype="10x",
    device=DEVICE,
    random_seed=TRAIN_SEED,
)


print("=" * 100)
print("SPATIALGLUE HLN-A1 SEED0 — TRAINING")
print("=" * 100)

print(
    "Device        :",
    DEVICE,
)

print(
    "Training seed :",
    TRAIN_SEED,
)

print(
    "Epochs        :",
    model.epochs,
)

print(
    "Weight factors:",
    model.weight_factors,
)


assert model.epochs == 200


output = model.train()


# ============================================================
# 4. Extract joint embedding
# ============================================================

assert (
    "SpatialGlue"
    in output
), output.keys()


embedding = np.asarray(
    output["SpatialGlue"]
)


assert (
    embedding.shape[0]
    == len(gt)
)

assert np.isfinite(
    embedding
).all()


print(
    "\nEmbedding shape :",
    embedding.shape,
)

print(
    "Embedding dtype :",
    embedding.dtype,
)

print(
    "Embedding finite:",
    np.isfinite(
        embedding
    ).all(),
)


# ============================================================
# 5. Save TRAINING output before clustering
# ============================================================

np.save(
    OUT_DIR
    / "embedding.npy",
    embedding,
)

np.save(
    OUT_DIR
    / "gt_labels.npy",
    gt,
)

np.save(
    OUT_DIR
    / "coords.npy",
    coords,
)


training_info = {
    "method":
        "SpatialGlue",

    "dataset":
        DATASET,

    "training_seed":
        TRAIN_SEED,

    "datatype":
        "10x",

    "epochs":
        int(model.epochs),

    "weight_factors":
        [
            float(x)
            for x in model.weight_factors
        ],

    "n_clusters":
        N_CLUSTERS,

    "n_spots":
        int(len(gt)),

    "embedding_shape":
        list(
            embedding.shape
        ),
}


with (
    OUT_DIR
    / "training_info.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        training_info,
        f,
        indent=2,
    )


print(
    "\nSaved training output:"
)

print(
    OUT_DIR
)

print(
    "\nPASS: SpatialGlue training completed."
)

print(
    "PASS: embedding saved before clustering."
)

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


SPATIALGLUE HLN-A1 SEED0 — TRAINING
Device        : cuda
Training seed : 0
Epochs        : 200
Weight factors: [1, 5, 1, 10]


  0%|          | 0/200 [00:00<?, ?it/s]/kaggle/working/SpatialGlue/SpatialGlue/model.py:212: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6)
100%|██████████| 200/200 [00:03<00:00, 59.19it/s]

Model training finished!


Embedding shape : (3484, 64)
Embedding dtype : float32
Embedding finite: True

Saved training output:
/kaggle/working/SpatialGlue_baseline/smoke/HLNA1_seed0

PASS: SpatialGlue training completed.
PASS: embedding saved before clustering.


Cell 13：官方 PCA20 + Rscript mclust + ARI/NM

In [18]:
# ============================================================
# Cell 13
# SpatialGlue HLN-A1 seed0 clustering + evaluation
#
# Saved SpatialGlue embedding
#        ↓
# official PCA (20D)
#        ↓
# R mclust
#   G = 10
#   modelNames = "EEE"
#   seed = 2020
#        ↓
# ARI / NMI
#
# Rscript file bridge is used only because the original
# rpy2 numpy2rpy bridge fails in the current environment.
# ============================================================

from pathlib import Path
import subprocess
import json

import numpy as np
import anndata as ad

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from SpatialGlue.preprocess import (
    pca,
)


# ============================================================
# 1. Frozen protocol
# ============================================================

DATASET = "HLN-A1"
TRAIN_SEED = 0
N_CLUSTERS = 10

MCLUST_SEED = 2020
MCLUST_MODEL = "EEE"
PCA_COMPONENTS = 20


OUT_DIR = (
    Path("/kaggle/working")
    / "SpatialGlue_baseline"
    / "smoke"
    / "HLNA1_seed0"
)


assert OUT_DIR.exists()


# ============================================================
# 2. Load saved training outputs
# ============================================================

embedding = np.load(
    OUT_DIR
    / "embedding.npy"
)

gt = np.load(
    OUT_DIR
    / "gt_labels.npy",
    allow_pickle=True,
)

coords = np.load(
    OUT_DIR
    / "coords.npy"
)


assert (
    embedding.shape[0]
    == len(gt)
)

assert np.isfinite(
    embedding
).all()


print("=" * 100)
print("LOAD SAVED SPATIALGLUE EMBEDDING")
print("=" * 100)

print(
    "Embedding:",
    embedding.shape,
)

print(
    "GT:",
    gt.shape,
)

print(
    "Coordinates:",
    coords.shape,
)


# ============================================================
# 3. Official SpatialGlue PCA step
#
# Equivalent to:
# clustering(..., use_pca=True, n_comps=20)
# ============================================================

tmp_adata = ad.AnnData(
    X=np.zeros(
        (
            embedding.shape[0],
            1,
        ),
        dtype=np.float32,
    )
)

tmp_adata.obsm[
    "SpatialGlue"
] = embedding


embedding_pca20 = pca(
    tmp_adata,
    use_reps="SpatialGlue",
    n_comps=PCA_COMPONENTS,
)


embedding_pca20 = np.asarray(
    embedding_pca20,
    dtype=np.float64,
)


assert (
    embedding_pca20.shape
    == (
        len(gt),
        PCA_COMPONENTS,
    )
)

assert np.isfinite(
    embedding_pca20
).all()


np.save(
    OUT_DIR
    / "embedding_pca20.npy",
    embedding_pca20,
)


print(
    "\nPCA embedding:",
    embedding_pca20.shape,
)


# ============================================================
# 4. Write PCA matrix for R
# ============================================================

BRIDGE_DIR = (
    OUT_DIR
    / "mclust_bridge"
)

BRIDGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


input_path = (
    BRIDGE_DIR
    / "pca20.tsv"
)

label_path = (
    BRIDGE_DIR
    / "mclust_labels.tsv"
)

r_script_path = (
    BRIDGE_DIR
    / "run_mclust.R"
)


np.savetxt(
    input_path,
    embedding_pca20,
    delimiter="\t",
    fmt="%.17g",
)


# ============================================================
# 5. Exact mclust script
# ============================================================

r_code = f'''
suppressPackageStartupMessages(
    library(mclust)
)

X <- as.matrix(
    read.table(
        "{input_path}",
        header = FALSE,
        sep = "\\t",
        check.names = FALSE
    )
)

storage.mode(X) <- "double"

cat(
    "R matrix dimensions:",
    nrow(X),
    "x",
    ncol(X),
    "\\n"
)

if (
    nrow(X) != {len(gt)}
    || ncol(X) != {PCA_COMPONENTS}
) {{
    stop(
        "Unexpected PCA matrix dimensions."
    )
}}

if (
    any(!is.finite(X))
) {{
    stop(
        "Non-finite values detected."
    )
}}

set.seed(
    {MCLUST_SEED}
)

fit <- Mclust(
    X,
    G = {N_CLUSTERS},
    modelNames = "{MCLUST_MODEL}"
)

labels <- fit$classification

if (
    length(labels)
    != nrow(X)
) {{
    stop(
        "Classification length mismatch."
    )
}}

write.table(
    labels,
    file = "{label_path}",
    row.names = FALSE,
    col.names = FALSE,
    quote = FALSE
)

cat(
    "mclust clusters:",
    length(unique(labels)),
    "\\n"
)

cat(
    "mclust model:",
    fit$modelName,
    "\\n"
)
'''


r_script_path.write_text(
    r_code,
    encoding="utf-8",
)


# ============================================================
# 6. Run mclust in R
# ============================================================

proc = subprocess.run(
    [
        "Rscript",
        str(r_script_path),
    ],
    text=True,
    capture_output=True,
)


print(
    "\n" + "=" * 100
)

print(
    "R MCLUST OUTPUT"
)

print(
    "=" * 100
)

print(
    proc.stdout
)


if proc.stderr.strip():

    print(
        "R stderr:"
    )

    print(
        proc.stderr
    )


assert (
    proc.returncode
    == 0
), (
    "R mclust failed."
)


assert label_path.exists()


# ============================================================
# 7. Load predictions
# ============================================================

pred = np.loadtxt(
    label_path,
    dtype=np.int64,
)

pred = np.asarray(
    pred
).reshape(-1)


assert (
    len(pred)
    == len(gt)
)


n_pred = len(
    np.unique(pred)
)


assert (
    n_pred
    == N_CLUSTERS
), (
    f"Expected {N_CLUSTERS} clusters, "
    f"got {n_pred}"
)


# ============================================================
# 8. Independent unified evaluation
# ============================================================

ari = adjusted_rand_score(
    gt,
    pred,
)

nmi = (
    normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )
)


print(
    "\n" + "=" * 100
)

print(
    "HLN-A1 SPATIALGLUE SEED0 RESULT"
)

print(
    "=" * 100
)

print(
    f"ARI = {ari:.12f}"
)

print(
    f"NMI = {nmi:.12f}"
)

print(
    "Predicted clusters =",
    n_pred,
)

print(
    "Target clusters    =",
    N_CLUSTERS,
)


# ============================================================
# 9. Save predictions + formal smoke metrics
# ============================================================

np.save(
    OUT_DIR
    / "pred_labels.npy",
    pred,
)


metrics = {
    "method":
        "SpatialGlue",

    "dataset":
        DATASET,

    "training_seed":
        TRAIN_SEED,

    "datatype":
        "10x",

    "epochs":
        200,

    "embedding":
        "SpatialGlue",

    "clustering":
        "mclust",

    "clustering_bridge":
        "Rscript_file_bridge",

    "use_pca":
        True,

    "pca_components":
        PCA_COMPONENTS,

    "mclust_seed":
        MCLUST_SEED,

    "mclust_modelNames":
        MCLUST_MODEL,

    "n_clusters":
        N_CLUSTERS,

    "ARI":
        float(ari),

    "NMI":
        float(nmi),

    "NMI_average_method":
        "max",
}


with (
    OUT_DIR
    / "metrics.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metrics,
        f,
        indent=2,
    )


# Exact R script also kept as evidence
shutil_required = False

print(
    "\nSaved:"
)

print(
    OUT_DIR
)

print(
    "\nPASS: HLN-A1 SpatialGlue "
    "seed0 clustering verified."
)

LOAD SAVED SPATIALGLUE EMBEDDING
Embedding: (3484, 64)
GT: (3484,)
Coordinates: (3484, 2)

PCA embedding: (3484, 20)

R MCLUST OUTPUT
R matrix dimensions: 3484 x 20 
mclust clusters: 10 
mclust model: EEE 


HLN-A1 SPATIALGLUE SEED0 RESULT
ARI = 0.242327874709
NMI = 0.345040869398
Predicted clusters = 10
Target clusters    = 10

Saved:
/kaggle/working/SpatialGlue_baseline/smoke/HLNA1_seed0

PASS: HLN-A1 SpatialGlue seed0 clustering verified.


Cell 14：把 Cell 13 的 mclust 过程封装成通用函数

In [19]:
# ============================================================
# Cell 14
# Reusable SpatialGlue downstream clustering
#
# SpatialGlue embedding
#        ↓
# PCA 20
#        ↓
# R mclust
#   modelNames = EEE
#   seed = 2020
#
# Uses Rscript file bridge because current rpy2 bridge fails.
# ============================================================

from pathlib import Path
import subprocess

import numpy as np
import anndata as ad

from SpatialGlue.preprocess import pca


def spatialglue_mclust(
    embedding,
    n_clusters,
    out_dir,
    tag,
    pca_components=20,
    mclust_seed=2020,
    model_names="EEE",
):

    out_dir = Path(out_dir)

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Input audit
    # --------------------------------------------------------

    embedding = np.asarray(
        embedding
    )


    assert embedding.ndim == 2

    assert np.isfinite(
        embedding
    ).all()


    # --------------------------------------------------------
    # Official PCA step
    # --------------------------------------------------------

    tmp = ad.AnnData(
        X=np.zeros(
            (
                embedding.shape[0],
                1,
            ),
            dtype=np.float32,
        )
    )


    tmp.obsm[
        "SpatialGlue"
    ] = embedding


    emb_pca = pca(
        tmp,
        use_reps="SpatialGlue",
        n_comps=pca_components,
    )


    emb_pca = np.asarray(
        emb_pca,
        dtype=np.float64,
    )


    assert (
        emb_pca.shape
        == (
            embedding.shape[0],
            pca_components,
        )
    )

    assert np.isfinite(
        emb_pca
    ).all()


    # --------------------------------------------------------
    # Save PCA representation
    # --------------------------------------------------------

    pca_path = (
        out_dir
        / "embedding_pca20.npy"
    )

    np.save(
        pca_path,
        emb_pca,
    )


    # --------------------------------------------------------
    # Bridge files
    # --------------------------------------------------------

    bridge_dir = (
        out_dir
        / "mclust_bridge"
    )

    bridge_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    input_path = (
        bridge_dir
        / f"{tag}_pca20.tsv"
    )

    label_path = (
        bridge_dir
        / f"{tag}_mclust_labels.tsv"
    )

    script_path = (
        bridge_dir
        / f"{tag}_mclust.R"
    )


    np.savetxt(
        input_path,
        emb_pca,
        delimiter="\t",
        fmt="%.17g",
    )


    # --------------------------------------------------------
    # Same mclust parameters as SpatialGlue
    # --------------------------------------------------------

    r_code = f'''
suppressPackageStartupMessages(
    library(mclust)
)

X <- as.matrix(
    read.table(
        "{input_path}",
        header = FALSE,
        sep = "\\t",
        check.names = FALSE
    )
)

storage.mode(X) <- "double"

cat(
    "R matrix dimensions:",
    nrow(X),
    "x",
    ncol(X),
    "\\n"
)

if (
    nrow(X) != {embedding.shape[0]}
    || ncol(X) != {pca_components}
) {{
    stop(
        "Unexpected PCA matrix dimensions."
    )
}}

if (
    any(!is.finite(X))
) {{
    stop(
        "Non-finite PCA values."
    )
}}

set.seed(
    {mclust_seed}
)

fit <- Mclust(
    X,
    G = {n_clusters},
    modelNames = "{model_names}"
)

labels <- fit$classification

if (
    length(labels)
    != nrow(X)
) {{
    stop(
        "Classification length mismatch."
    )
}}

write.table(
    labels,
    file = "{label_path}",
    row.names = FALSE,
    col.names = FALSE,
    quote = FALSE
)

cat(
    "mclust clusters:",
    length(unique(labels)),
    "\\n"
)

cat(
    "mclust model:",
    fit$modelName,
    "\\n"
)
'''


    script_path.write_text(
        r_code,
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Execute R
    # --------------------------------------------------------

    proc = subprocess.run(
        [
            "Rscript",
            str(script_path),
        ],
        text=True,
        capture_output=True,
    )


    print(
        proc.stdout
    )


    if proc.stderr.strip():

        print(
            "R stderr:"
        )

        print(
            proc.stderr
        )


    assert (
        proc.returncode
        == 0
    ), "R mclust failed."


    assert (
        label_path.exists()
    )


    pred = np.loadtxt(
        label_path,
        dtype=np.int64,
    )

    pred = np.asarray(
        pred
    ).reshape(-1)


    assert (
        len(pred)
        == embedding.shape[0]
    )


    assert (
        len(
            np.unique(pred)
        )
        == n_clusters
    ), (
        f"Expected {n_clusters} clusters, "
        f"got {len(np.unique(pred))}"
    )


    return (
        pred,
        emb_pca,
    )


print(
    "PASS: reusable SpatialGlue "
    "mclust function defined."
)

PASS: reusable SpatialGlue mclust function defined.


Cell 15：E18.5 seed0 完整 smoke

In [22]:
# ============================================================
# Cell 15 - FINAL
# SpatialGlue E18.5 seed0 smoke test
#
# Dataset:
#   E18.5
#   RNA + ATAC
#
# Frozen benchmark population:
#   2129 spots
#   K = 14
#
# SpatialGlue protocol:
#   datatype = Spatial-epigenome-transcriptome
#   RNA feat  = PCA50
#   ATAC feat = LSI50
#   epochs    = 1600
#   weights   = [1, 5, 1, 1]
#
# Downstream:
#   SpatialGlue embedding
#        -> PCA20
#        -> R mclust
#           G = 14
#           modelNames = EEE
#           seed = 2020
#
# Evaluation:
#   ARI = sklearn
#   NMI = average_method="max"
# ============================================================

from pathlib import Path
import json
import gc

import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from SpatialGlue.SpatialGlue_pyG import (
    Train_SpatialGlue,
)

from SpatialGlue.preprocess import (
    fix_seed,
)


# ============================================================
# 1. Frozen protocol
# ============================================================

DATASET = "E18.5"

TRAIN_SEED = 0

N_CLUSTERS = 14

EXPECTED_N_SPOTS = 2129

EXPECTED_FEAT_DIM = 50

EXPECTED_EMBED_DIM = 64


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


assert DEVICE.type == "cuda", (
    "GPU is not enabled."
)


OUT_DIR = (
    Path("/kaggle/working")
    / "SpatialGlue_baseline"
    / "smoke"
    / "E185_seed0"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 110)
print("SPATIALGLUE E18.5 SEED0 SMOKE TEST")
print("=" * 110)

print(
    "Dataset      :",
    DATASET,
)

print(
    "Training seed:",
    TRAIN_SEED,
)

print(
    "Target K     :",
    N_CLUSTERS,
)

print(
    "Device       :",
    DEVICE,
)


# ============================================================
# 2. Preprocess E18.5
# ============================================================

fix_seed(
    TRAIN_SEED
)


print(
    "\n" + "=" * 110
)

print(
    "STEP 1 — E18.5 PREPROCESSING"
)

print(
    "=" * 110
)


prepared = (
    prepare_spatialglue_dataset(
        DATASET
    )
)


adata_rna = (
    prepared[
        "adata_omics1"
    ]
)

adata_atac = (
    prepared[
        "adata_omics2"
    ]
)

data = (
    prepared[
        "data"
    ]
)

gt = np.asarray(
    prepared[
        "labels"
    ]
)

coords = np.asarray(
    adata_rna.obsm[
        "spatial"
    ]
)


# ============================================================
# 3. Preprocessing audit
# ============================================================

rna_feat = np.asarray(
    adata_rna.obsm[
        "feat"
    ]
)

atac_feat = np.asarray(
    adata_atac.obsm[
        "feat"
    ]
)


print(
    "\nSpots       :",
    adata_rna.n_obs,
)

print(
    "GT clusters :",
    len(
        np.unique(gt)
    ),
)

print(
    "RNA feat    :",
    rna_feat.shape,
)

print(
    "ATAC feat   :",
    atac_feat.shape,
)

print(
    "Spatial     :",
    coords.shape,
)


assert (
    adata_rna.n_obs
    == EXPECTED_N_SPOTS
), (
    f"Expected {EXPECTED_N_SPOTS} RNA spots, "
    f"got {adata_rna.n_obs}"
)


assert (
    adata_atac.n_obs
    == EXPECTED_N_SPOTS
), (
    f"Expected {EXPECTED_N_SPOTS} ATAC spots, "
    f"got {adata_atac.n_obs}"
)


assert (
    len(gt)
    == EXPECTED_N_SPOTS
)


assert (
    len(
        np.unique(gt)
    )
    == N_CLUSTERS
)


assert (
    rna_feat.shape
    == (
        EXPECTED_N_SPOTS,
        EXPECTED_FEAT_DIM,
    )
), (
    f"Unexpected RNA feat shape: "
    f"{rna_feat.shape}"
)


assert (
    atac_feat.shape
    == (
        EXPECTED_N_SPOTS,
        EXPECTED_FEAT_DIM,
    )
), (
    f"Unexpected ATAC feat shape: "
    f"{atac_feat.shape}"
)


assert (
    coords.shape
    == (
        EXPECTED_N_SPOTS,
        2,
    )
)


assert np.isfinite(
    rna_feat
).all()


assert np.isfinite(
    atac_feat
).all()


assert np.isfinite(
    coords
).all()


assert np.array_equal(
    np.asarray(
        adata_rna.obs_names
    ),
    np.asarray(
        adata_atac.obs_names
    ),
)


print(
    "\nPASS: E18.5 preprocessing verified."
)

print(
    "PASS: all 2129 benchmark spots preserved."
)


# ============================================================
# 4. Train official SpatialGlue model
# ============================================================

fix_seed(
    TRAIN_SEED
)


model = Train_SpatialGlue(
    data,
    datatype=(
        "Spatial-epigenome-transcriptome"
    ),
    device=DEVICE,
    random_seed=TRAIN_SEED,
)


print(
    "\n" + "=" * 110
)

print(
    "STEP 2 — SPATIALGLUE TRAINING"
)

print(
    "=" * 110
)


print(
    "Device        :",
    DEVICE,
)

print(
    "Training seed :",
    TRAIN_SEED,
)

print(
    "Epochs        :",
    model.epochs,
)

print(
    "Weight factors:",
    model.weight_factors,
)


assert (
    int(model.epochs)
    == 1600
), (
    f"Expected 1600 epochs, "
    f"got {model.epochs}"
)


assert (
    list(
        model.weight_factors
    )
    == [
        1,
        5,
        1,
        1,
    ]
), (
    "Unexpected SpatialGlue weight factors: "
    f"{model.weight_factors}"
)


output = model.train()


print(
    "\nPASS: SpatialGlue training completed."
)


# ============================================================
# 5. Extract joint embedding
# ============================================================

assert (
    "SpatialGlue"
    in output
), (
    f"'SpatialGlue' missing from output. "
    f"Keys = {list(output.keys())}"
)


embedding = np.asarray(
    output[
        "SpatialGlue"
    ]
)


print(
    "\nEmbedding shape :",
    embedding.shape,
)

print(
    "Embedding dtype :",
    embedding.dtype,
)

print(
    "Embedding finite:",
    np.isfinite(
        embedding
    ).all(),
)


assert (
    embedding.shape
    == (
        EXPECTED_N_SPOTS,
        EXPECTED_EMBED_DIM,
    )
), (
    f"Unexpected embedding shape: "
    f"{embedding.shape}"
)


assert np.isfinite(
    embedding
).all()


# ============================================================
# 6. Save training outputs BEFORE clustering
# ============================================================

np.save(
    OUT_DIR
    / "embedding.npy",
    embedding,
)


np.save(
    OUT_DIR
    / "gt_labels.npy",
    gt,
)


np.save(
    OUT_DIR
    / "coords.npy",
    coords,
)


training_info = {

    "method":
        "SpatialGlue",

    "dataset":
        DATASET,

    "training_seed":
        TRAIN_SEED,

    "datatype":
        "Spatial-epigenome-transcriptome",

    "n_spots":
        EXPECTED_N_SPOTS,

    "n_clusters":
        N_CLUSTERS,

    "rna_feature_dim":
        int(
            rna_feat.shape[1]
        ),

    "atac_feature_dim":
        int(
            atac_feat.shape[1]
        ),

    "epochs":
        int(
            model.epochs
        ),

    "weight_factors":
        [
            float(x)
            for x
            in model.weight_factors
        ],

    "embedding_shape":
        list(
            embedding.shape
        ),
}


with (
    OUT_DIR
    / "training_info.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        training_info,
        f,
        indent=2,
    )


print(
    "\nTraining evidence saved:"
)

print(
    OUT_DIR
)


# ============================================================
# 7. PCA20 -> R mclust
#
# Uses spatialglue_mclust() defined in Cell 14.
# ============================================================

assert (
    "spatialglue_mclust"
    in globals()
), (
    "spatialglue_mclust() is not defined. "
    "Please run Cell 14 first."
)


print(
    "\n" + "=" * 110
)

print(
    "STEP 3 — PCA20 + R MCLUST"
)

print(
    "=" * 110
)


pred, embedding_pca20 = (
    spatialglue_mclust(
        embedding=embedding,
        n_clusters=N_CLUSTERS,
        out_dir=OUT_DIR,
        tag="E185_seed0",
        pca_components=20,
        mclust_seed=2020,
        model_names="EEE",
    )
)


pred = np.asarray(
    pred
).reshape(-1)


assert (
    len(pred)
    == EXPECTED_N_SPOTS
)


n_pred_clusters = len(
    np.unique(pred)
)


assert (
    n_pred_clusters
    == N_CLUSTERS
), (
    f"Expected {N_CLUSTERS} clusters, "
    f"got {n_pred_clusters}"
)


assert (
    embedding_pca20.shape
    == (
        EXPECTED_N_SPOTS,
        20,
    )
)


# ============================================================
# 8. Unified independent evaluation
# ============================================================

ari = adjusted_rand_score(
    gt,
    pred,
)


nmi = normalized_mutual_info_score(
    gt,
    pred,
    average_method="max",
)


print(
    "\n" + "=" * 110
)

print(
    "E18.5 SPATIALGLUE SEED0 RESULT"
)

print(
    "=" * 110
)


print(
    f"ARI = {ari:.12f}"
)

print(
    f"NMI = {nmi:.12f}"
)

print(
    "Predicted clusters =",
    n_pred_clusters,
)

print(
    "Target clusters    =",
    N_CLUSTERS,
)


# ============================================================
# 9. Save predictions + final smoke metrics
# ============================================================

np.save(
    OUT_DIR
    / "pred_labels.npy",
    pred,
)


metrics = {

    "method":
        "SpatialGlue",

    "dataset":
        DATASET,

    "training_seed":
        TRAIN_SEED,

    "datatype":
        "Spatial-epigenome-transcriptome",

    "benchmark_population":
        "all_spots",

    "n_spots":
        EXPECTED_N_SPOTS,

    "epochs":
        int(
            model.epochs
        ),

    "weight_factors":
        [
            float(x)
            for x
            in model.weight_factors
        ],

    "embedding":
        "SpatialGlue",

    "embedding_dim":
        int(
            embedding.shape[1]
        ),

    "clustering":
        "mclust",

    "clustering_bridge":
        "Rscript_file_bridge",

    "use_pca":
        True,

    "pca_components":
        20,

    "mclust_seed":
        2020,

    "mclust_modelNames":
        "EEE",

    "n_clusters":
        N_CLUSTERS,

    "predicted_clusters":
        int(
            n_pred_clusters
        ),

    "ARI":
        float(ari),

    "NMI":
        float(nmi),

    "NMI_average_method":
        "max",
}


with (
    OUT_DIR
    / "metrics.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metrics,
        f,
        indent=2,
    )


print(
    "\nSaved:"
)

print(
    OUT_DIR
)


print(
    "\nPASS: E18.5 SpatialGlue "
    "seed0 smoke test complete."
)

print(
    "PASS: 2129/2129 spots evaluated."
)

print(
    "PASS: PCA20 + mclust "
    "clustering verified."
)


# ============================================================
# 10. Cleanup
# ============================================================

del model
del output
del prepared
del data

gc.collect()

torch.cuda.empty_cache()


print(
    "\nGPU cache cleared."
)

SPATIALGLUE E18.5 SEED0 SMOKE TEST
Dataset      : E18.5
Training seed: 0
Target K     : 14
Device       : cuda

STEP 1 — E18.5 PREPROCESSING

PREPARING SPATIALGLUE DATASET: E18.5
Initial spots: 2129
E18.5 metadata adapter: PASS

Preprocessing mode: RNA + ATAC / Spatial-epigenome-transcriptome


/kaggle/working/SpatialGlue/SpatialGlue/preprocess.py:209: RuntimeWarning: divide by zero encountered in divide
  idf = X.shape[0] / X.sum(axis=0)



Feature audit:
  RNA feat: (2129, 50)
  ATAC feat: (2129, 50)

[E18.5] spots preserved: 2129/2129
[E18.5] GT clusters: 14

Constructing SpatialGlue graphs...

--------------------------------------------------------------------------------------------------------------
E18.5 PREPARATION COMPLETE
--------------------------------------------------------------------------------------------------------------
Spots: 2129
K: 14
RNA feat: (2129, 50)
ATAC feat: (2129, 50)
Spatial: (2129, 2)

PASS: SpatialGlue dataset preparation verified.

Spots       : 2129
GT clusters : 14
RNA feat    : (2129, 50)
ATAC feat   : (2129, 50)
Spatial     : (2129, 2)

PASS: E18.5 preprocessing verified.
PASS: all 2129 benchmark spots preserved.

STEP 2 — SPATIALGLUE TRAINING
Device        : cuda
Training seed : 0
Epochs        : 1600
Weight factors: [1, 5, 1, 1]


  0%|          | 0/1600 [00:00<?, ?it/s]/kaggle/working/SpatialGlue/SpatialGlue/model.py:212: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6)
100%|██████████| 1600/1600 [00:25<00:00, 62.09it/s]


Model training finished!


PASS: SpatialGlue training completed.

Embedding shape : (2129, 64)
Embedding dtype : float32
Embedding finite: True

Training evidence saved:
/kaggle/working/SpatialGlue_baseline/smoke/E185_seed0

STEP 3 — PCA20 + R MCLUST
R matrix dimensions: 2129 x 20 
mclust clusters: 14 
mclust model: EEE 


E18.5 SPATIALGLUE SEED0 RESULT
ARI = 0.371588337483
NMI = 0.557302612418
Predicted clusters = 14
Target clusters    = 14

Saved:
/kaggle/working/SpatialGlue_baseline/smoke/E185_seed0

PASS: E18.5 SpatialGlue seed0 smoke test complete.
PASS: 2129/2129 spots evaluated.
PASS: PCA20 + mclust clustering verified.

GPU cache cleared.


Cell 16：冻结 SpatialGlue 正式实验协议

In [23]:
# ============================================================
# Cell 16
# Freeze SpatialGlue formal benchmark protocol
# ============================================================

from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import torch
import sklearn
import scanpy as sc
import anndata


FORMAL_ROOT = (
    Path("/kaggle/working")
    / "SpatialGlue_baseline"
    / "formal_10seeds"
)

FORMAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FORMAL_SEEDS = list(
    range(10)
)


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED_PROTOCOL = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
        "datatype": "10x",
        "epochs": 200,
        "weights": [1, 5, 1, 10],
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
        "datatype": "10x",
        "epochs": 200,
        "weights": [1, 5, 1, 10],
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
        "datatype":
            "Spatial-epigenome-transcriptome",
        "epochs": 1600,
        "weights": [1, 5, 1, 1],
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
        "datatype":
            "Spatial-epigenome-transcriptome",
        "epochs": 1600,
        "weights": [1, 5, 1, 1],
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
        "datatype":
            "Spatial-epigenome-transcriptome",
        "epochs": 1600,
        "weights": [1, 5, 1, 1],
    },
}


# ------------------------------------------------------------
# Git commit
# ------------------------------------------------------------

try:

    spatialglue_commit = (
        subprocess.check_output(
            [
                "git",
                "-C",
                str(SPATIALGLUE_ROOT),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        )
        .strip()
    )

except Exception:

    spatialglue_commit = "unknown"


protocol = {

    "method":
        "SpatialGlue",

    "seeds":
        FORMAL_SEEDS,

    "datasets":
        EXPECTED_PROTOCOL,

    "benchmark_population":
        "frozen_all_spots",

    "preprocessing_note":
        (
            "Feature-level SpatialGlue preprocessing is retained. "
            "Observation-level filtering is disabled so all methods "
            "are evaluated on the same benchmark spots."
        ),

    "embedding":
        "SpatialGlue joint embedding",

    "clustering":
        "mclust",

    "clustering_use_pca":
        True,

    "clustering_pca_components":
        20,

    "mclust_seed":
        2020,

    "mclust_modelNames":
        "EEE",

    "NMI_average_method":
        "max",

    "std_ddof":
        0,

    "SpatialGlue_git_commit":
        spatialglue_commit,

    "software": {

        "python":
            sys.version.split()[0],

        "torch":
            torch.__version__,

        "numpy":
            np.__version__,

        "sklearn":
            sklearn.__version__,

        "scanpy":
            sc.__version__,

        "anndata":
            anndata.__version__,
    },
}


with (
    FORMAL_ROOT
    / "protocol.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )


print("=" * 100)
print("SPATIALGLUE FORMAL PROTOCOL")
print("=" * 100)

print(
    "Output root:",
    FORMAL_ROOT,
)

print(
    "Seeds:",
    FORMAL_SEEDS,
)

print(
    "SpatialGlue commit:",
    spatialglue_commit,
)

print(
    "\nPASS: formal protocol frozen."
)

SPATIALGLUE FORMAL PROTOCOL
Output root: /kaggle/working/SpatialGlue_baseline/formal_10seeds
Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
SpatialGlue commit: 7c976d811d27ace51ce47ae0ad94a068a7d222fa

PASS: formal protocol frozen.


Cell 17：正式 5 数据集 × 10 seeds runner

In [24]:
# ============================================================
# Cell 17
# SpatialGlue formal benchmark
#
# 5 datasets x 10 training seeds = 50 runs
#
# Per dataset:
#   preprocess ONCE
#   train seeds 0..9
#
# Per run:
#   SpatialGlue embedding
#       -> PCA20
#       -> R mclust
#       -> ARI / NMI(max)
#
# Supports resume from existing metrics.json files.
# ============================================================

from pathlib import Path
import copy
import json
import traceback
import gc
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from SpatialGlue.SpatialGlue_pyG import (
    Train_SpatialGlue,
)

from SpatialGlue.preprocess import (
    fix_seed,
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

assert DEVICE.type == "cuda"


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


SKIP_EXISTING = True


all_rows = []


# ============================================================
# Helper: load existing successful run
# ============================================================

def load_existing_metrics(
    metrics_path,
):

    try:

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        required = [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
            "n_clusters",
            "predicted_clusters",
        ]


        if not all(
            key in m
            for key in required
        ):

            return None


        return m

    except Exception:

        return None


# ============================================================
# Dataset loop
# ============================================================

for dataset_name in DATASET_ORDER:

    expected = (
        EXPECTED_PROTOCOL[
            dataset_name
        ]
    )


    dataset_dir = (
        FORMAL_ROOT
        / DATASET_FOLDER[
            dataset_name
        ]
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "\n\n" + "#" * 110
    )

    print(
        f"# DATASET: {dataset_name}"
    )

    print(
        "#" * 110
    )


    # ========================================================
    # 1. Preprocess ONCE for this dataset
    # ========================================================

    print(
        "\nPreparing dataset once..."
    )


    fix_seed(0)


    prepared = (
        prepare_spatialglue_dataset(
            dataset_name
        )
    )


    adata_omics1 = (
        prepared[
            "adata_omics1"
        ]
    )

    adata_omics2 = (
        prepared[
            "adata_omics2"
        ]
    )

    data_base = (
        prepared[
            "data"
        ]
    )

    gt = np.asarray(
        prepared[
            "labels"
        ]
    )

    coords = np.asarray(
        adata_omics1.obsm[
            "spatial"
        ]
    )


    feat1 = np.asarray(
        adata_omics1.obsm[
            "feat"
        ]
    )

    feat2 = np.asarray(
        adata_omics2.obsm[
            "feat"
        ]
    )


    # ========================================================
    # 2. Dataset-level audit
    # ========================================================

    assert (
        len(gt)
        == expected[
            "n_spots"
        ]
    )


    assert (
        len(
            np.unique(gt)
        )
        == expected["K"]
    )


    assert (
        coords.shape
        == (
            expected[
                "n_spots"
            ],
            2,
        )
    )


    assert np.isfinite(
        feat1
    ).all()


    assert np.isfinite(
        feat2
    ).all()


    dataset_info = {

        "dataset":
            dataset_name,

        "n_spots":
            int(len(gt)),

        "n_clusters":
            int(
                expected["K"]
            ),

        "datatype":
            expected[
                "datatype"
            ],

        "omics1_feature_shape":
            list(
                feat1.shape
            ),

        "omics2_feature_shape":
            list(
                feat2.shape
            ),

        "coordinates_shape":
            list(
                coords.shape
            ),
    }


    with (
        dataset_dir
        / "dataset_preprocessing.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            dataset_info,
            f,
            indent=2,
        )


    # Shared GT/coords for audit
    np.save(
        dataset_dir
        / "gt_labels.npy",
        gt,
    )

    np.save(
        dataset_dir
        / "coords.npy",
        coords,
    )


    print(
        "Dataset preprocessing: PASS"
    )


    # ========================================================
    # 3. Seed loop
    # ========================================================

    for seed in FORMAL_SEEDS:

        run_dir = (
            dataset_dir
            / f"seed{seed}"
        )

        run_dir.mkdir(
            parents=True,
            exist_ok=True,
        )


        metrics_path = (
            run_dir
            / "metrics.json"
        )


        # ----------------------------------------------------
        # Resume
        # ----------------------------------------------------

        if (
            SKIP_EXISTING
            and metrics_path.exists()
        ):

            existing = (
                load_existing_metrics(
                    metrics_path
                )
            )


            if existing is not None:

                print(
                    f"[SKIP] "
                    f"{dataset_name} seed={seed} "
                    f"ARI={existing['ARI']:.6f} "
                    f"NMI={existing['NMI']:.6f}"
                )

                all_rows.append(
                    existing
                )

                continue


        print(
            "\n" + "-" * 100
        )

        print(
            f"RUN: {dataset_name} | "
            f"seed={seed}"
        )

        print(
            "-" * 100
        )


        start_time = time.time()


        try:

            # =================================================
            # Training seed
            # =================================================

            fix_seed(
                seed
            )


            # Defensive copy so a model run cannot
            # mutate the dataset-level graph object.
            data_run = copy.deepcopy(
                data_base
            )


            model = Train_SpatialGlue(
                data_run,
                datatype=expected[
                    "datatype"
                ],
                device=DEVICE,
                random_seed=seed,
            )


            # =================================================
            # Protocol assertions
            # =================================================

            assert (
                int(model.epochs)
                == expected[
                    "epochs"
                ]
            )


            assert (
                list(
                    model.weight_factors
                )
                == expected[
                    "weights"
                ]
            )


            print(
                "epochs =",
                model.epochs,
            )

            print(
                "weights =",
                model.weight_factors,
            )


            # =================================================
            # Train
            # =================================================

            output = model.train()


            embedding = np.asarray(
                output[
                    "SpatialGlue"
                ]
            )


            assert (
                embedding.shape[0]
                == expected[
                    "n_spots"
                ]
            )


            assert np.isfinite(
                embedding
            ).all()


            # =================================================
            # Save embedding BEFORE clustering
            # =================================================

            np.save(
                run_dir
                / "embedding.npy",
                embedding,
            )


            # =================================================
            # Official downstream clustering
            # =================================================

            pred, emb_pca20 = (
                spatialglue_mclust(
                    embedding=embedding,
                    n_clusters=expected[
                        "K"
                    ],
                    out_dir=run_dir,
                    tag=(
                        f"{DATASET_FOLDER[dataset_name]}"
                        f"_seed{seed}"
                    ),
                    pca_components=20,
                    mclust_seed=2020,
                    model_names="EEE",
                )
            )


            pred = np.asarray(
                pred
            ).reshape(-1)


            n_pred = len(
                np.unique(pred)
            )


            assert (
                len(pred)
                == len(gt)
            )


            assert (
                n_pred
                == expected[
                    "K"
                ]
            )


            # =================================================
            # Unified evaluation
            # =================================================

            ari = (
                adjusted_rand_score(
                    gt,
                    pred,
                )
            )


            nmi = (
                normalized_mutual_info_score(
                    gt,
                    pred,
                    average_method="max",
                )
            )


            elapsed = (
                time.time()
                - start_time
            )


            # =================================================
            # Save predictions
            # =================================================

            np.save(
                run_dir
                / "pred_labels.npy",
                pred,
            )


            # =================================================
            # Metrics
            # =================================================

            metrics = {

                "method":
                    "SpatialGlue",

                "dataset":
                    dataset_name,

                "training_seed":
                    int(seed),

                "datatype":
                    expected[
                        "datatype"
                    ],

                "benchmark_population":
                    "frozen_all_spots",

                "n_spots":
                    int(len(gt)),

                "epochs":
                    int(
                        model.epochs
                    ),

                "weight_factors":
                    [
                        float(x)
                        for x
                        in model.weight_factors
                    ],

                "embedding_dim":
                    int(
                        embedding.shape[1]
                    ),

                "clustering":
                    "mclust",

                "clustering_bridge":
                    "Rscript_file_bridge",

                "use_pca":
                    True,

                "pca_components":
                    20,

                "mclust_seed":
                    2020,

                "mclust_modelNames":
                    "EEE",

                "n_clusters":
                    int(
                        expected["K"]
                    ),

                "predicted_clusters":
                    int(
                        n_pred
                    ),

                "ARI":
                    float(ari),

                "NMI":
                    float(nmi),

                "NMI_average_method":
                    "max",

                "runtime_seconds":
                    float(elapsed),
            }


            with metrics_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    metrics,
                    f,
                    indent=2,
                )


            all_rows.append(
                metrics
            )


            print(
                f"\nRESULT "
                f"{dataset_name} "
                f"seed={seed}"
            )

            print(
                f"ARI = {ari:.12f}"
            )

            print(
                f"NMI = {nmi:.12f}"
            )

            print(
                f"time = "
                f"{elapsed:.1f}s"
            )


            # =================================================
            # Cleanup
            # =================================================

            del model
            del output
            del embedding
            del pred
            del emb_pca20
            del data_run

            gc.collect()

            torch.cuda.empty_cache()


        except Exception as e:

            error_info = {

                "method":
                    "SpatialGlue",

                "dataset":
                    dataset_name,

                "training_seed":
                    int(seed),

                "error":
                    repr(e),

                "traceback":
                    traceback.format_exc(),
            }


            with (
                run_dir
                / "ERROR.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    error_info,
                    f,
                    indent=2,
                )


            print(
                "\nFAILED:"
            )

            print(
                repr(e)
            )


            gc.collect()

            torch.cuda.empty_cache()


    # ========================================================
    # Dataset cleanup
    # ========================================================

    del prepared
    del adata_omics1
    del adata_omics2
    del data_base
    del feat1
    del feat2

    gc.collect()

    torch.cuda.empty_cache()


# ============================================================
# 4. Build RAW table
# ============================================================

raw_df = pd.DataFrame(
    all_rows
)


raw_path = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_RAW.csv"
)


raw_df.to_csv(
    raw_path,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "RAW RESULTS"
)

print(
    "=" * 110
)

print(
    raw_df[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 5. Audit run counts
# ============================================================

counts = (
    raw_df
    .groupby(
        "dataset"
    )
    .size()
)


print(
    "\nRun counts:"
)

print(
    counts
)


for dataset_name in DATASET_ORDER:

    assert (
        int(
            counts.get(
                dataset_name,
                0,
            )
        )
        == 10
    ), (
        f"{dataset_name}: "
        "does not have 10 successful runs."
    )


assert (
    len(raw_df)
    == 50
)


# ============================================================
# 6. Summary mean ± population std (ddof=0)
# ============================================================

summary_rows = []


for dataset_name in DATASET_ORDER:

    part = (
        raw_df[
            raw_df[
                "dataset"
            ]
            == dataset_name
        ]
        .sort_values(
            "training_seed"
        )
    )


    assert (
        len(part)
        == 10
    )


    summary_rows.append(
        {

            "dataset":
                dataset_name,

            "n_runs":
                10,

            "ARI_mean":
                float(
                    part[
                        "ARI"
                    ].mean()
                ),

            "ARI_std":
                float(
                    part[
                        "ARI"
                    ].std(
                        ddof=0
                    )
                ),

            "NMI_mean":
                float(
                    part[
                        "NMI"
                    ].mean()
                ),

            "NMI_std":
                float(
                    part[
                        "NMI"
                    ].std(
                        ddof=0
                    )
                ),
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_path = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_SUMMARY.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "SPATIALGLUE FORMAL SUMMARY"
)

print(
    "=" * 110
)


for _, row in summary_df.iterrows():

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | "
        f"NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
    )


print(
    "\nSaved RAW:"
)

print(
    raw_path
)

print(
    "\nSaved SUMMARY:"
)

print(
    summary_path
)


print(
    "\nPASS: 50/50 SpatialGlue "
    "formal runs completed."
)



##############################################################################################################
# DATASET: HLN-A1
##############################################################################################################

Preparing dataset once...

PREPARING SPATIALGLUE DATASET: HLN-A1


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Initial spots: 3484

Preprocessing mode: RNA + ADT / 10x


/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)



Feature audit:
  RNA feat: (3484, 30)
  ADT feat: (3484, 30)

[HLN-A1] spots preserved: 3484/3484
[HLN-A1] GT clusters: 10

Constructing SpatialGlue graphs...

--------------------------------------------------------------------------------------------------------------
HLN-A1 PREPARATION COMPLETE
--------------------------------------------------------------------------------------------------------------
Spots: 3484
K: 10
RNA feat: (3484, 30)
ADT feat: (3484, 30)
Spatial: (3484, 2)

PASS: SpatialGlue dataset preparation verified.
Dataset preprocessing: PASS

----------------------------------------------------------------------------------------------------
RUN: HLN-A1 | seed=0
----------------------------------------------------------------------------------------------------
epochs = 200
weights = [1, 5, 1, 10]


...

Model training finished!

R matrix dimensions: 2248 x 20 
mclust clusters: 16 
mclust model: EEE 


RESULT S2-E18 seed=7
ARI = 0.290019354601
NMI = 0.457443562965
time = 37.6s

----------------------------------------------------------------------------------------------------
RUN: S2-E18 | seed=8
----------------------------------------------------------------------------------------------------
epochs = 1600
weights = [1, 5, 1, 1]


  0%|          | 0/1600 [00:00<?, ?it/s]/kaggle/working/SpatialGlue/SpatialGlue/model.py:212: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6)
100%|██████████| 1600/1600 [00:25<00:00, 61.87it/s]


Model training finished!

R matrix dimensions: 2248 x 20 
mclust clusters: 16 
mclust model: EEE 


RESULT S2-E18 seed=8
ARI = 0.359770457028
NMI = 0.471705178940
time = 33.5s

----------------------------------------------------------------------------------------------------
RUN: S2-E18 | seed=9
----------------------------------------------------------------------------------------------------
epochs = 1600
weights = [1, 5, 1, 1]


  0%|          | 0/1600 [00:00<?, ?it/s]/kaggle/working/SpatialGlue/SpatialGlue/model.py:212: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6)
100%|██████████| 1600/1600 [00:25<00:00, 61.68it/s]


Model training finished!

R matrix dimensions: 2248 x 20 
mclust clusters: 16 
mclust model: EEE 


RESULT S2-E18 seed=9
ARI = 0.393456540193
NMI = 0.469896985369
time = 34.5s

RAW RESULTS
dataset  training_seed      ARI      NMI
 HLN-A1              0 0.242328 0.345041
 HLN-A1              1 0.244600 0.352964
 HLN-A1              2 0.253379 0.350376
 HLN-A1              3 0.251379 0.331433
 HLN-A1              4 0.233392 0.344244
 HLN-A1              5 0.217831 0.325591
 HLN-A1              6 0.293486 0.369569
 HLN-A1              7 0.217323 0.323740
 HLN-A1              8 0.262876 0.353494
 HLN-A1              9 0.242484 0.345115
 HLN-D1              0 0.226890 0.310246
 HLN-D1              1 0.202194 0.311246
 HLN-D1              2 0.211763 0.307712
 HLN-D1              3 0.165176 0.283081
 HLN-D1              4 0.169594 0.291474
 HLN-D1              5 0.163358 0.287968
 HLN-D1              6 0.152470 0.279564
 HLN-D1              7 0.212633 0.314036
 HLN-D1              8 0.185132 

Cell 18：50/50 独立审计

In [25]:
# ============================================================
# Cell 18
# SpatialGlue formal 50/50 independent audit
#
# Verify:
#   1. exactly 5 datasets
#   2. exactly seeds 0..9
#   3. every run has metrics + predictions
#   4. recompute ARI / NMI from GT + prediction
#   5. verify metrics.json
#   6. verify RAW.csv
#   7. independently recompute mean/std (ddof=0)
#   8. verify SUMMARY.csv
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FORMAL_ROOT = Path(
    "/kaggle/working/SpatialGlue_baseline/formal_10seeds"
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED = {
    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
        "epochs": 200,
    },
    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
        "epochs": 200,
    },
    "E18.5": {
        "n_spots": 2129,
        "K": 14,
        "epochs": 1600,
    },
    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
        "epochs": 1600,
    },
    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
        "epochs": 1600,
    },
}


RAW_PATH = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_RAW.csv"
)

SUMMARY_PATH = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_SUMMARY.csv"
)


assert RAW_PATH.exists()
assert SUMMARY_PATH.exists()


raw_saved = pd.read_csv(
    RAW_PATH
)

summary_saved = pd.read_csv(
    SUMMARY_PATH
)


audit_rows = []


print("=" * 110)
print("SPATIALGLUE 50-RUN INDEPENDENT AUDIT")
print("=" * 110)


# ============================================================
# 1. Recompute every run
# ============================================================

for dataset in DATASET_ORDER:

    dataset_dir = (
        FORMAL_ROOT
        / DATASET_FOLDER[dataset]
    )


    gt_path = (
        dataset_dir
        / "gt_labels.npy"
    )


    assert gt_path.exists(), gt_path


    gt = np.load(
        gt_path,
        allow_pickle=True,
    )


    assert (
        len(gt)
        == EXPECTED[dataset]["n_spots"]
    )


    assert (
        len(np.unique(gt))
        == EXPECTED[dataset]["K"]
    )


    found_seeds = []


    for seed in range(10):

        run_dir = (
            dataset_dir
            / f"seed{seed}"
        )


        metrics_path = (
            run_dir
            / "metrics.json"
        )

        pred_path = (
            run_dir
            / "pred_labels.npy"
        )

        emb_path = (
            run_dir
            / "embedding.npy"
        )

        pca_path = (
            run_dir
            / "embedding_pca20.npy"
        )


        assert run_dir.exists(), run_dir
        assert metrics_path.exists(), metrics_path
        assert pred_path.exists(), pred_path
        assert emb_path.exists(), emb_path
        assert pca_path.exists(), pca_path


        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            metrics = json.load(f)


        pred = np.load(
            pred_path,
            allow_pickle=True,
        )


        embedding = np.load(
            emb_path
        )

        embedding_pca = np.load(
            pca_path
        )


        # ----------------------------------------------------
        # Structural audit
        # ----------------------------------------------------

        assert len(pred) == len(gt)

        assert (
            len(np.unique(pred))
            == EXPECTED[dataset]["K"]
        )

        assert (
            embedding.shape[0]
            == len(gt)
        )

        assert (
            embedding_pca.shape
            == (
                len(gt),
                20,
            )
        )

        assert np.isfinite(
            embedding
        ).all()

        assert np.isfinite(
            embedding_pca
        ).all()


        # ----------------------------------------------------
        # Protocol audit
        # ----------------------------------------------------

        assert (
            metrics["dataset"]
            == dataset
        )

        assert (
            int(metrics["training_seed"])
            == seed
        )

        assert (
            int(metrics["epochs"])
            == EXPECTED[dataset]["epochs"]
        )

        assert (
            int(metrics["n_clusters"])
            == EXPECTED[dataset]["K"]
        )

        assert (
            int(metrics["predicted_clusters"])
            == EXPECTED[dataset]["K"]
        )

        assert (
            metrics["clustering"]
            == "mclust"
        )

        assert (
            metrics["mclust_modelNames"]
            == "EEE"
        )

        assert (
            int(metrics["mclust_seed"])
            == 2020
        )

        assert (
            int(metrics["pca_components"])
            == 20
        )

        assert (
            metrics["NMI_average_method"]
            == "max"
        )


        # ----------------------------------------------------
        # Independent metric recomputation
        # ----------------------------------------------------

        ari_recomputed = (
            adjusted_rand_score(
                gt,
                pred,
            )
        )

        nmi_recomputed = (
            normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )
        )


        assert np.isclose(
            ari_recomputed,
            float(metrics["ARI"]),
            rtol=0,
            atol=1e-12,
        ), (
            f"{dataset} seed{seed}: "
            "ARI mismatch"
        )


        assert np.isclose(
            nmi_recomputed,
            float(metrics["NMI"]),
            rtol=0,
            atol=1e-12,
        ), (
            f"{dataset} seed{seed}: "
            "NMI mismatch"
        )


        audit_rows.append(
            {
                "dataset":
                    dataset,

                "training_seed":
                    seed,

                "ARI":
                    float(
                        ari_recomputed
                    ),

                "NMI":
                    float(
                        nmi_recomputed
                    ),
            }
        )


        found_seeds.append(
            seed
        )


    assert (
        found_seeds
        == list(range(10))
    )


    print(
        f"{dataset:8s}: "
        f"10/10 runs PASS"
    )


# ============================================================
# 2. Build independently reconstructed RAW table
# ============================================================

audit_raw = pd.DataFrame(
    audit_rows
)


assert len(audit_raw) == 50


audit_raw = (
    audit_raw
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


raw_check = (
    raw_saved[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    audit_raw[
        [
            "dataset",
            "training_seed",
        ]
    ]
    .equals(
        raw_check[
            [
                "dataset",
                "training_seed",
            ]
        ]
    )
)


assert np.allclose(
    audit_raw["ARI"],
    raw_check["ARI"],
    rtol=0,
    atol=1e-12,
)


assert np.allclose(
    audit_raw["NMI"],
    raw_check["NMI"],
    rtol=0,
    atol=1e-12,
)


print(
    "\nPASS: RAW CSV matches "
    "independent metric recomputation."
)


# ============================================================
# 3. Independently reconstruct summary
# ============================================================

audit_summary_rows = []


for dataset in DATASET_ORDER:

    part = audit_raw[
        audit_raw["dataset"]
        == dataset
    ]


    assert len(part) == 10


    audit_summary_rows.append(
        {
            "dataset":
                dataset,

            "n_runs":
                10,

            "ARI_mean":
                float(
                    part["ARI"].mean()
                ),

            "ARI_std":
                float(
                    part["ARI"].std(
                        ddof=0
                    )
                ),

            "NMI_mean":
                float(
                    part["NMI"].mean()
                ),

            "NMI_std":
                float(
                    part["NMI"].std(
                        ddof=0
                    )
                ),
        }
    )


audit_summary = pd.DataFrame(
    audit_summary_rows
)


# ============================================================
# 4. Compare against saved SUMMARY
# ============================================================

summary_check = (
    summary_saved
    .set_index("dataset")
    .loc[DATASET_ORDER]
    .reset_index()
)


for col in [
    "ARI_mean",
    "ARI_std",
    "NMI_mean",
    "NMI_std",
]:

    assert np.allclose(
        audit_summary[col],
        summary_check[col],
        rtol=0,
        atol=1e-12,
    ), (
        f"SUMMARY mismatch: {col}"
    )


assert np.array_equal(
    audit_summary[
        "n_runs"
    ].to_numpy(),
    np.full(
        5,
        10,
    ),
)


# ============================================================
# 5. Save audit outputs
# ============================================================

AUDIT_RAW_PATH = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_AUDIT_RAW.csv"
)

AUDIT_SUMMARY_PATH = (
    FORMAL_ROOT
    / "SpatialGlue_5datasets_10seeds_AUDIT_SUMMARY.csv"
)


audit_raw.to_csv(
    AUDIT_RAW_PATH,
    index=False,
)

audit_summary.to_csv(
    AUDIT_SUMMARY_PATH,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "INDEPENDENT SUMMARY"
)

print(
    "=" * 110
)


for _, row in (
    audit_summary.iterrows()
):

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
    )


print(
    "\nPASS: metrics.json = predictions = RAW.csv = SUMMARY.csv"
)

print(
    "PASS: std uses ddof=0"
)

print(
    "PASS: seeds 0-9 complete for all five datasets"
)

print(
    "PASS: 50/50 independent audit verified"
)

SPATIALGLUE 50-RUN INDEPENDENT AUDIT
HLN-A1  : 10/10 runs PASS
HLN-D1  : 10/10 runs PASS
E18.5   : 10/10 runs PASS
S2-E15  : 10/10 runs PASS
S2-E18  : 10/10 runs PASS

PASS: RAW CSV matches independent metric recomputation.

INDEPENDENT SUMMARY
HLN-A1   | ARI 0.245908 ± 0.021037 | NMI 0.344157 ± 0.013320
HLN-D1   | ARI 0.186465 ± 0.023994 | NMI 0.299326 ± 0.012260
E18.5    | ARI 0.503367 ± 0.071251 | NMI 0.570431 ± 0.017553
S2-E15   | ARI 0.435312 ± 0.011079 | NMI 0.575966 ± 0.009656
S2-E18   | ARI 0.338322 ± 0.048174 | NMI 0.468577 ± 0.013155

PASS: metrics.json = predictions = RAW.csv = SUMMARY.csv
PASS: std uses ddof=0
PASS: seeds 0-9 complete for all five datasets
PASS: 50/50 independent audit verified


Cell 19：生成最终归档 ZIP + SHA256

In [26]:
# ============================================================
# Cell 19
# Archive SpatialGlue formal benchmark
#
# Includes:
#   protocol
#   RAW / SUMMARY
#   independent audit
#   GT / coordinates
#   dataset preprocessing records
#   each run:
#       metrics
#       predictions
#       embedding
#       PCA20 embedding
#       exact R mclust script (if present)
#
# Excludes:
#   temporary PCA TSV bridge files
# ============================================================

from pathlib import Path
import shutil
import hashlib
import json


FORMAL_ROOT = Path(
    "/kaggle/working/SpatialGlue_baseline/formal_10seeds"
)


ARCHIVE_NAME = (
    "SpatialGlue_5datasets_10seeds_FORMAL_FINAL"
)

STAGE_DIR = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)

ZIP_BASE = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)


# ============================================================
# 1. Clean staging directory
# ============================================================

if STAGE_DIR.exists():

    shutil.rmtree(
        STAGE_DIR
    )


STAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Copy top-level formal records
# ============================================================

top_files = [
    "protocol.json",
    "SpatialGlue_5datasets_10seeds_RAW.csv",
    "SpatialGlue_5datasets_10seeds_SUMMARY.csv",
    "SpatialGlue_5datasets_10seeds_AUDIT_RAW.csv",
    "SpatialGlue_5datasets_10seeds_AUDIT_SUMMARY.csv",
]


for name in top_files:

    src = (
        FORMAL_ROOT
        / name
    )

    assert src.exists(), src

    shutil.copy2(
        src,
        STAGE_DIR
        / name,
    )


# ============================================================
# 3. Dataset information
# ============================================================

DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


for dataset, folder in (
    DATASET_FOLDER.items()
):

    src_dataset = (
        FORMAL_ROOT
        / folder
    )

    dst_dataset = (
        STAGE_DIR
        / folder
    )

    dst_dataset.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Shared dataset-level files
    # --------------------------------------------------------

    for name in [
        "dataset_preprocessing.json",
        "gt_labels.npy",
        "coords.npy",
    ]:

        src = (
            src_dataset
            / name
        )

        assert src.exists(), src

        shutil.copy2(
            src,
            dst_dataset
            / name,
        )


    # --------------------------------------------------------
    # 10 runs
    # --------------------------------------------------------

    for seed in range(10):

        src_run = (
            src_dataset
            / f"seed{seed}"
        )

        dst_run = (
            dst_dataset
            / f"seed{seed}"
        )

        dst_run.mkdir(
            parents=True,
            exist_ok=True,
        )


        essential_files = [
            "metrics.json",
            "pred_labels.npy",
            "embedding.npy",
            "embedding_pca20.npy",
        ]


        for name in essential_files:

            src = (
                src_run
                / name
            )

            assert src.exists(), src

            shutil.copy2(
                src,
                dst_run
                / name,
            )


        # ----------------------------------------------------
        # Preserve exact R script if available
        # ----------------------------------------------------

        bridge_dir = (
            src_run
            / "mclust_bridge"
        )


        if bridge_dir.exists():

            r_scripts = list(
                bridge_dir.glob(
                    "*.R"
                )
            )


            if r_scripts:

                dst_bridge = (
                    dst_run
                    / "mclust_bridge"
                )

                dst_bridge.mkdir(
                    parents=True,
                    exist_ok=True,
                )


                for r_script in r_scripts:

                    shutil.copy2(
                        r_script,
                        dst_bridge
                        / r_script.name,
                    )


# ============================================================
# 4. Add archive README
# ============================================================

README_TEXT = """\
SpatialGlue formal baseline archive

Protocol:
- Five datasets
- Training seeds: 0-9
- 10 runs per dataset
- 50 total successful runs
- SpatialGlue official datatype-dependent training epochs
- SpatialGlue joint embedding
- PCA: 20 dimensions before clustering
- Clustering: R mclust
- mclust modelNames: EEE
- mclust seed: 2020
- ARI: sklearn adjusted_rand_score
- NMI: sklearn normalized_mutual_info_score(average_method="max")
- Standard deviation: population std, ddof=0

Benchmark-population adaptation:
Observation-level filtering was disabled so every method is evaluated
on exactly the same benchmark spots. Feature-level preprocessing was
retained.

E18.5 metadata adaptation:
spatial coordinates = [array_col, array_row]
ground truth = Combined_Clusters_annotation

Compatibility adaptations:
- SciPy-compatible CLR conversion uses .toarray() instead of legacy .A
- Rscript file bridge replaces failing rpy2 numpy2rpy conversion
- SpatialGlue mathematical/model protocol is otherwise retained

Temporary PCA TSV bridge files are intentionally excluded.
Saved embedding_pca20.npy is retained.
"""


(
    STAGE_DIR
    / "README_ARCHIVE.txt"
).write_text(
    README_TEXT,
    encoding="utf-8",
)


# ============================================================
# 5. SHA256 manifest
# ============================================================

def sha256_file(
    path,
):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


manifest_lines = []


for path in sorted(
    STAGE_DIR.rglob("*")
):

    if not path.is_file():
        continue


    # Avoid hashing the manifest into itself
    if (
        path.name
        == "SHA256SUMS.txt"
    ):
        continue


    relative = path.relative_to(
        STAGE_DIR
    )


    digest = sha256_file(
        path
    )


    manifest_lines.append(
        f"{digest}  {relative}"
    )


MANIFEST_PATH = (
    STAGE_DIR
    / "SHA256SUMS.txt"
)


MANIFEST_PATH.write_text(
    "\n".join(
        manifest_lines
    )
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 6. Final structural audit
# ============================================================

metrics_files = list(
    STAGE_DIR.rglob(
        "metrics.json"
    )
)

pred_files = list(
    STAGE_DIR.rglob(
        "pred_labels.npy"
    )
)

embedding_files = list(
    STAGE_DIR.rglob(
        "embedding.npy"
    )
)


assert len(metrics_files) == 50
assert len(pred_files) == 50
assert len(embedding_files) == 50


print("=" * 110)
print("ARCHIVE STRUCTURAL AUDIT")
print("=" * 110)

print(
    "metrics.json:",
    len(metrics_files),
)

print(
    "pred_labels.npy:",
    len(pred_files),
)

print(
    "embedding.npy:",
    len(embedding_files),
)

print(
    "SHA256 entries:",
    len(manifest_lines),
)


# ============================================================
# 7. Create ZIP
# ============================================================

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=STAGE_DIR.parent,
    base_dir=STAGE_DIR.name,
)


zip_path = Path(
    zip_path
)


zip_size_mb = (
    zip_path.stat().st_size
    / 1024
    / 1024
)


print(
    "\nZIP:"
)

print(
    zip_path
)

print(
    f"ZIP size: {zip_size_mb:.2f} MB"
)


print(
    "\nPASS: SpatialGlue formal archive created."
)

print(
    "PASS: 50 metrics + 50 predictions + "
    "50 embeddings archived."
)

print(
    "PASS: SHA256 manifest created."
)

ARCHIVE STRUCTURAL AUDIT
metrics.json: 50
pred_labels.npy: 50
embedding.npy: 50
SHA256 entries: 271

ZIP:
/kaggle/working/SpatialGlue_5datasets_10seeds_FORMAL_FINAL.zip
ZIP size: 41.12 MB

PASS: SpatialGlue formal archive created.
PASS: 50 metrics + 50 predictions + 50 embeddings archived.
PASS: SHA256 manifest created.
